## Modeliing of Venting Dynamics in Controlled Environments
### Karla Peña Ramírez

The Simonyi Telescope at the Vera C. Rubin Observatory is the largest astronomical imager in the world. Its image quality is directly related with three main factors: optics, dome thermals and atmospheric perturbations. In this study I model the venting dynamics inside the dome closure, which itself is a controlled environment working under a complex engineered thermal‑management scheme. 


In [ ]:
#Setting packages
import asyncio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
import time
import warnings
import seaborn as sns
import datetime as dt
from scipy import stats
from matplotlib.dates import DateFormatter
from astropy.table import Table, join
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score
from scipy.cluster.hierarchy import dendrogram, linkage
from lsst.utils.packages import getEnvironmentPackages
from IPython.display import Markdown, display
#%matplotlib widget


#####IQ metric
import astropy.units as u
from astropy.coordinates import get_sun, AltAz, EarthLocation
from astropy.time import Time, TimeDelta
from datetime import datetime, timezone, timedelta
import galsim
import pytz
import matplotlib.dates as mdates

%matplotlib inline

from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
import os

import matplotlib as mpl
from matplotlib.colors import Normalize


from lsst.daf.butler import Butler 
#from tqdm.notebook import tqdm


from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import Ridge
from sklearn.model_selection import ShuffleSplit
import warnings

# Ignore the many warning messages from ``merge_packed_time_series``
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning)

#-------Clients
from lsst_efd_client import EfdClient
client = EfdClient("usdf_efd")

#from lsst.summit.utils.efdUtils import makeEfdClient
#client = makeEfdClient()

#from lsst.summit.utils import ConsDbClient
#os.environ["no_proxy"] += ",.consdb"
#url="http://consdb-pq.consdb:8080/consdb"
#consdb=ConsDbClient(url)
#-------Clients

def print_session_info():
    # Time info
    print(f"# Session Info on {time.strftime('%Y-%m-%d at %H:%M:%S %Z', time.localtime(time.time()))}\n")

    # Python info
    print(f"## Python Interpreter\n\nVersion: {sys.version}  \nExecutable: {sys.executable}\n")

    # LSST info
    packages = getEnvironmentPackages(True)
    dev_packages = {"lsst_distrib": packages["lsst_distrib"]}
    dev_packages.update({k: v.split("@")[0] for k, v in packages.items() if "LOCAL" in v})
    print("## Science Pipelines\n\n" + "\n".join(f"{k:<20} {v}" for k, v in dev_packages.items()))

Data retrieval for the on-sky images:

In [ ]:
#Identify the repo where all the data lives and the latest consolidated processing collection
#Run > butler query-collections /repo/main "*DRP*w_2025_0*" | grep CHAINED
repo = '/repo/main'
instrument = "LSSTCam"
collection = 'LSSTCam/raw/all','LSSTCam/runs/nightlyValidation'

butler = Butler(repo, collections=collection)
registry = butler.registry




In [ ]:
#Query the metadata for the **`exposure`** dimension, limiting the results to this particular instrument and range of days of observation:
instrument = 'LSSTCam'
day_obs_start = 20250701
day_obs_end = 20250921
query="instrument='%s' AND day_obs>=%d AND day_obs<=%d" % (instrument, day_obs_start, day_obs_end)
results = registry.queryDimensionRecords('exposure', where=query)

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Stop executing if there are no results returned:
n_results = results.count()

if n_results <= 0:
    raise StopExecution
else:
    print("""There are %d results returned from querying the butler for instrument %s between dates %d and %d (inclusive).""" % 
          (n_results, instrument, day_obs_start, day_obs_end))

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Instantiate a pandas `DataFrame` with useful columns available in the `exposure` dimension:
df_exp = pd.DataFrame(columns=['id', 'obs_id','day_obs', 'seq_num',
                                    'time_start','time_end' ,'type', 'reason', 
                                    'target','filter','zenith_angle',
                                    'expos','ra','dec','skyangle',
                                    'azimuth','zenith','science_program',
                                    'jd','mjd'])

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Read the query results into the new pandas `DataFrame`:
for count, info in enumerate(results):
    
    try:

        df_exp.loc[count] = [info.id, info.obs_id, info.day_obs, info.seq_num, 
                                  info.timespan.begin.utc.iso,
                                  info.timespan.end.utc.iso, 
                                  info.observation_type, info.observation_reason, info.target_name, 
                                  info.physical_filter, info.zenith_angle, 
                                  info.exposure_time,info.tracking_ra, info.tracking_dec, 
                                  info.sky_angle,info.azimuth ,info.zenith_angle, 
                                  info.science_program, info.timespan.begin.jd, info.timespan.begin.mjd]

    except:
    
        print(">>>   Unexpected error:", sys.exc_info()[0])
        info_timespan_begin_to_string = "2021-01-01 00:00:00.00"
        info_timespan_end_to_string = "2051-01-01 00:00:00.00"
        info_timespan_begin_jd = 0
        info_timespan_begin_mjd = 0
        df_exp.loc[count] = [info.id, info.obs_id, info.day_obs, info.seq_num, 
                                  pd.to_datetime(info_timespan_begin_to_string),
                                  pd.to_datetime(info_timespan_end_to_string), 
                                  info.observation_type, info.observation_reason, info.target_name, 
                                  info.physical_filter, info.zenith_angle, 
                                  info.exposure_time,info.tracking_ra, info.tracking_dec, 
                                  info.sky_angle,info.azimuth ,info.zenith_angle, 
                                  info.science_program, info_timespan_begin_jd, info_timespan_begin_mjd ]    

Taking into account only SV data. Filtering data under the nitial aligment (acq) and system checkouts.

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Clean-up the dataframe
#Re-cast the `id`, `day_obs`, and `seq_num` rows as `int`'s:
df_exp = df_exp.astype({"id": int,'day_obs': int,'seq_num':int})
# Replace `NaN`'s in the `ra` and `dec` columns with zero.  
df_exp['ra'] = df_exp['ra'].fillna(0)
df_exp['dec'] = df_exp['dec'].fillna(0)
#Select only on-sky data
df_exp.type.unique()

#df_open = df_exp[(df_exp.type == 'science') | (df_exp.type == 'cwfs') | (df_exp.type == 'focus') | (df_exp.type == 'acq')| (df_exp.type == 'flat')]
#df_open = df_exp[(df_exp.type == 'science') | (df_exp.type == 'cwfs') | (df_exp.type == 'focus') | (df_exp.type == 'acq')]
df_open = df_exp[(df_exp.type == 'science') | (df_exp.type == 'cwfs') | (df_exp.type == 'focus') | ((df_exp.type == 'acq') & (df_exp.science_program == 'BLOCK-T539'))]
#df_open = df_exp[(df_exp.type == 'science')]

In [ ]:
##Visualization.
df_ = df_open.copy()
df_['day_obs'] = pd.to_numeric(df_['day_obs'], errors='coerce').astype('Int64')
df_ = df_.dropna(subset=['day_obs']).copy()
df_['day_obs'] = df_['day_obs'].astype(int)

#Stacked bar per night by type.
pivot = (df_.groupby(['day_obs', 'type'])
           .size()
           .unstack(fill_value=0)
           .sort_index())

#Better labels.
pivot.index = pd.to_datetime(pivot.index.astype(str), format='%Y%m%d')

fig, ax = plt.subplots(figsize=(25, 5))
pivot.plot(kind='bar', stacked=True, ax=ax, width=0.9)
ax.set_title('On-sky exposures per night by type')
ax.set_xlabel('Night [UTC]')
ax.set_ylabel('# Exposures')
ax.grid(True, axis='y', alpha=0.3)

# Reduce x-tick clutter if many nights
#if len(pivot) > 20:
#    step = max(1, len(pivot)//20)
#    for i, label in enumerate(ax.get_xticklabels()):
#        if i % step != 0:
#            label.set_visible(False)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

##Report
print("\nExposure counts by day_obs:")
print(df_.groupby(['day_obs', 'type']).size())
print(df_.groupby(['day_obs', 'type']).size().sum())
   


Get the initial/final times of data taken on a given night 

In [ ]:
#Group data by dayobs identifying the extremme dates for the exposures.
result_on_sky = df_open.groupby('day_obs').agg({
    'time_start': 'min',
    'time_end': 'max'
}).reset_index()

result_on_sky

Amplifying the period telemetry

In [ ]:
result_on_sky['time_start'] = pd.to_datetime(result_on_sky['day_obs'].astype(str)) + pd.Timedelta(hours=19, minutes=30)
result_on_sky

In [ ]:
day_obs_start = 20250701   ####------>>>>> HERE
day_obs_end = 20250921   ####------>>>>> HERE

days = pd.date_range(pd.to_datetime(day_obs_start, format='%Y%m%d'),
                     pd.to_datetime(day_obs_end, format='%Y%m%d'))
result_on_sky = pd.DataFrame({
    "day_obs": days.strftime("%Y%m%d").astype("int64"),
    "time_start": (days + pd.Timedelta(hours=19, minutes=30)).strftime("%Y-%m-%d %H:%M"),
    "time_end": (days + pd.Timedelta(days=1, hours=12)).strftime("%Y-%m-%d %H:%M"),
})

print(result_on_sky.dtypes)
result_on_sky

In [ ]:
def read_time_data():
    df = result_on_sky
    df['time_start'] = pd.to_datetime(df['time_start'])
    df['time_end'] = pd.to_datetime(df['time_end'])
    return df
    
def get_time_range(df, day_obs):
    day_data = df[df['day_obs'] == day_obs].iloc[0]
    return day_data['time_start'], day_data['time_end']

Telemetry retrieval at 1min sampling:

In [ ]:
###ESS
###ESS inside dome
###Temperature
async def get_temperature_113_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get ESS temperature data for in-dome(M1M3) for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS temperature_113_Dome
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 113
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)

###Pressure
async def get_pressure_113_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get ESS pressure data for in-dome(M1M3) for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(pressureItem0) AS pressure_113_Dome
    FROM "lsst.sal.ESS.pressure"
    WHERE salIndex = 113
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)

###RH
async def get_RH_113_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get ESS RH data for in-dome(M1M3) for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(relativeHumidityItem) AS RH_113_Dome
    FROM "lsst.sal.ESS.relativeHumidity"
    WHERE salIndex = 113
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)



###ESS outside dome
###Temperature
async def get_temperature_301_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get ESS temperature data for Outside Dome for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(temperatureItem0) AS temperature_301_outDome
    FROM "lsst.sal.ESS.temperature"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)

####Pressure
async def get_pressure_301_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get pressure data for Outside Dome for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(pressureItem0) AS pressure_301_outDome
    FROM "lsst.sal.ESS.pressure"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)    

####Relative Humidity
async def get_RH_301_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get relative humidity data for Outside Dome for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(relativeHumidityItem) AS RH_301_outDome
    FROM "lsst.sal.ESS.relativeHumidity"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)



###Environmental general  
###ESS dewpoint
async def get_dewPoint_301_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get ESS data for dewpoint for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(dewPointItem) AS dewPoint_301
    FROM "lsst.sal.ESS.dewPoint"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)
  
####Sun Altitude
async def get_Sun_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get sun altitude data for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(sunAltitude) AS SunAltitude
    FROM "lsst.sal.MTPtg.skyEnvironment"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)

####Wind speed
async def get_wind_speed_301_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get wind speed data for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(speed) AS WindSpeed_301
    FROM "lsst.sal.ESS.airFlow"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


####Wind direction
async def get_wind_direction_301_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get wind direction data for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(direction) AS WindDirection_301
    FROM "lsst.sal.ESS.airFlow"
    WHERE salIndex = 301
    AND time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)



#####Dome Shutter
async def get_dome_shutter_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get dome shutter data for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(positionActual0) AS Dome_Shutter_Actual0,
        mean(positionActual1) AS Dome_Shutter_Actual1
    FROM "lsst.sal.MTDome.apertureShutter"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


#####Dome Azimuth
async def get_dome_azimuth_range(client, start_day_obs, end_day_obs, sampling="1m"):
    """Get dome azimuth data for a date range."""
    df = read_time_data()
    start_time, _ = get_time_range(df, int(start_day_obs))
    _, end_time = get_time_range(df, int(end_day_obs))
    
    query = f"""
    SELECT 
        mean(positionActual) AS Dome_Azimuth_Actual
    FROM "lsst.sal.MTDome.azimuth"
    WHERE time > '{start_time.isoformat()}Z'
    AND time < '{end_time.isoformat()}Z'
    GROUP BY time({sampling})
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)



In [ ]:
start_date = result_on_sky['day_obs'][0].astype(str)
start_date
end_date = result_on_sky['day_obs'].iloc[-1].astype(str)
end_date
result_on_sky

In [ ]:
# Get a range of data
start_date = '20250701'
end_date = '20250921'
sampling = '8m'   ####<<<<<<<<<<<----------- SAMPLING


#ESS temperature data and telemetry required
temp_113 = await get_temperature_113_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # Temp Dome
pressure_113 = await get_pressure_113_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # Pressure Dome
rh_113 = await get_RH_113_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # RH Dome

temp_301 = await get_temperature_301_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # Temp OutDome
pressure_301 = await get_pressure_301_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # Pressure OutDome
rh_301 = await get_RH_301_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # RH OutDome

dewPoint_301 = await get_dewPoint_301_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # dewPoint
SunAltitude = await get_Sun_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # Sun Altitute
WindSpeed_301 = await get_wind_speed_301_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # Wind SPEED
WindDirection_301 = await get_wind_direction_301_range(client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)  # Wind DIRECTION
DomeShutter = await get_dome_shutter_range (client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)   #Dome Shutter
DomeAzimuth = await get_dome_azimuth_range (client, sampling = sampling, start_day_obs = start_date, end_day_obs = end_date)   #Dome Azimuth

#Single data object to check the trends
data_objects = [
temp_113['temperature_113_Dome'],
pressure_113['pressure_113_Dome'],
rh_113['RH_113_Dome'],
temp_301['temperature_301_outDome'],
pressure_301['pressure_301_outDome'],
rh_301['RH_301_outDome'],
dewPoint_301['dewPoint_301'],
SunAltitude['SunAltitude'],
WindSpeed_301['WindSpeed_301'],
WindDirection_301['WindDirection_301'],
DomeShutter['Dome_Shutter_Actual0'],
DomeShutter['Dome_Shutter_Actual1'],
DomeAzimuth['Dome_Azimuth_Actual']
]

df = pd.concat(data_objects, axis=1)
df.shape


Visual data exploration 

In [ ]:
#Raw trends.
fig, ax = plt.subplots(1, 1, figsize=(18, 9), sharex=True)
plt.title('Trending Temperatures and Dewpoint')
ax = temp_113['temperature_113_Dome'].plot(linewidth=1, label='temp_113_Dome')
ax = temp_301['temperature_301_outDome'].plot(linewidth=1, label='temperature_301_outDome')
ax = dewPoint_301['dewPoint_301'].plot(linewidth=1, label='dewPoint_301')
plt.ylabel('Temperature [C]')
plt.legend(loc="best")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(18, 9), sharex=True)
plt.title('Trending Pressures')
ax = pressure_113['pressure_113_Dome'].plot(linewidth=1, label='pressure_113_Dome')
ax = pressure_301['pressure_301_outDome'].plot(linewidth=1, label='pressure_301_outDome')
plt.ylabel('Pressure [Bar?]')
plt.legend(loc="best")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(18, 9), sharex=True)
plt.title('Trending RH')
ax = rh_113['RH_113_Dome'].plot(linewidth=1, label='RH_113_Dome')
ax = rh_301['RH_301_outDome'].plot(linewidth=1, label='RH_301_outDome')
plt.ylabel('RH [%]')
plt.legend(loc="best")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(18, 9), sharex=True)
plt.title('Trending Sun Altitude')
ax = SunAltitude['SunAltitude'].plot(linewidth=1, label='SunAltitude')
plt.ylabel('Sun Altitude [deg]')
plt.legend(loc="best")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(18, 9), sharex=True)
plt.title('Trending Wind Speed')
ax = WindSpeed_301['WindSpeed_301'].plot(linewidth=1, label='WindSpeed_301')
plt.ylabel('Wind Speed [m/s]')
plt.legend(loc="best")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(18, 9), sharex=True)
plt.title('Trending Dome Shutter')
ax = DomeShutter['Dome_Shutter_Actual0'].plot(linewidth=1, label='Dome0')
ax = DomeShutter['Dome_Shutter_Actual1'].plot(linewidth=1, label='Dome1')
plt.ylim(0,100)
plt.ylabel('Dome [%]')
plt.legend(loc="best")
plt.show()

Let's see the relation of DeltaT and DeltaH with IQ. This can be used to see if the venting procedure is introducing heat to the dome or not. Also it will stablish if afternoon venting conditions will suffice. 

In [ ]:
#### Assisted by gpt-5 and claude4.5sonet.
#################Units and normalization.
def normalize_units_robust(df):
    out = df.copy()
    #RH telemetry is in percentages.
    #RH: % -> fraction.
    for col in out.columns:
        if col.lower().startswith('rh_'):
            med = out[col].dropna().median()
            if med is not None and med > 2.0:
                out[col] = out[col] / 100.0
            out[col] = out[col].clip(0.0, 1.0)
    #Constants for H need kPa. It seems telemetry came at 1e5 units. Pa.
    #Pressure -> kPa.
    def to_kPa(series):
        med = series.dropna().median()
        if med is None or not np.isfinite(med):
            return pd.Series(73.9, index=series.index)  # At ~2600 m, pressure is around 70kPa.
        if 2_000 < med < 300_000:   # Pa
            return series / 1000.0
        if 500 < med < 1200:        # hPa (mbar)
            return series / 10.0
        if 50 < med < 120:          # kPa
            return series
        if 0.5 < med < 2.0:         # bar
            return series * 100.0
        # Unknown: fallback
        return pd.Series(73.9, index=series.index)

    for pcol in ['pressure_113_Dome', 'pressure_301_outDome']:
        if pcol in out.columns:
            out[pcol + '_kPa'] = to_kPa(out[pcol])

    return out

#########Equations for DeltaH.
##Saturation vapor pressure over liquid water f(T). T needs to be in C. The output pressure is in kPa.
##Here we used the Magnus relation which is valid in the T=[-45,60]C. And alternative is the Hyland-Wexler relation.
def es_kPa(T_C):
    return 0.61094 * np.exp((17.625 * T_C) / (T_C + 243.04))

###Humidity ratio w f(T,RH,P). The equation is: w = 0.62198 * e / (P - e), with e = RH * es(T).
def humidity_ratio_from_T_RH_P(T_C, RH_frac, P_kPa):
    e = RH_frac * es_kPa(T_C)
    return 0.62198 * e / np.maximum(P_kPa - e, 1e-6)

###Humidity ratio w f(T,P) from dewpoint: at dewpoint RH=1 -> e = es(Td).
def humidity_ratio_from_Td_P(Td_C, P_kPa):
    e = es_kPa(Td_C)
    return 0.62198 * e / np.maximum(P_kPa - e, 1e-6)

####Moist-air enthalpy per kg dry air (ASHRAE approx) f(T,w) so f(T,RH,P) or f(T,P).
###h_dry = 1.006*T + w*(2501 + 1.86*T) [kJ/kg_dry]
def enthalpy_kJ_per_kg_dry(T_C, w):
    return 1.006*T_C + w*(2501.0 + 1.86*T_C)

###Effective moist-air specific heat per kg of moist air f(w) so f(T,RH,P) or f(T,P), using constant cp at near-ambient.
###The /(1+w) factor is to keep all in moist air.
###c_p,moist = (c_pa + w*c_pv) / (1 + w)
###c_pa ≈ 1.006, c_pv ≈ 1.86  [kJ/(kg·K)]    
def cp_moist_kJ_per_kg_moistK(w):
    return (1.006 + 1.86*w) / (1.0 + w)

###########Apply the equations to the data.
def compute_equations(df, smooth_roll_min=10, vent_persist_min=5):
    ###Apply the normalization function.
    out = normalize_units_robust(df).copy()
    #Aliases to calculate the functions.
    #In-dome telemetry.
    T_in  = out['temperature_113_Dome']
    RH_in = out['RH_113_Dome']
    #In case we change the detector and we do not have pressure_XXX_Dome the default value would be 73.9 kPa.
    P_in  = out.get('pressure_113_Dome_kPa', pd.Series(73.9, index=out.index))
    #Out-dome telemetry.
    T_out  = out['temperature_301_outDome']
    RH_out = out.get('RH_301_outDome', None)
    #In case there are available data of outside pressure the default value would be 73.9 kPa.
    P_out  = out.get('pressure_301_outDome_kPa', pd.Series(73.9, index=out.index))
    Td_out = out['dewPoint_301']

    ###Functions calculated.
    #Humidity ratios w f(T,RH,P).
    w_in  = humidity_ratio_from_T_RH_P(T_in, RH_in, P_in)
    w_out = humidity_ratio_from_Td_P(Td_out, P_out)
    if RH_out is not None:
        w_out = w_out.fillna(humidity_ratio_from_T_RH_P(T_out, RH_out, P_out))

    #Enthalpy per kg dry air.
    h_in_dry  = enthalpy_kJ_per_kg_dry(T_in,  w_in)
    h_out_dry = enthalpy_kJ_per_kg_dry(T_out, w_out)

    #Convert enthalpy to per kg *moist air*.
    h_in_moist  = h_in_dry  / (1.0 + w_in)
    h_out_moist = h_out_dry / (1.0 + w_out)

    ###Data storage and deltas.
    #Store the results as columns in the dataframe.
    out['w_in'] = w_in
    out['w_out'] = w_out
    out['h_in_kJkgdry']  = h_in_dry
    out['h_out_kJkgdry'] = h_out_dry
    out['h_in_kJkgmoist']  = h_in_moist
    out['h_out_kJkgmoist'] = h_out_moist
    #Δh and ΔT
    out['dhe_kJkg_moist'] = h_in_moist - h_out_moist  #>0 means outside is energetically cooler.
    out['dTe_C']          = T_in - T_out #>0 means outside is cooler than inside.
    #Condensation margin: change here in case we want a colder surface to check condensation.
    out['dew_margin_C']   = T_in - Td_out

    ###Temporal data: smoothing.
    #Smooth (sm) the temporal data using the rolling function applying the mean with a window given by smooth_roll_min.
    win = max(int(smooth_roll_min), 5)     #<<<<<<<<<<----------- SAMPLING
    for c in [
        'h_in_kJkgmoist','h_out_kJkgmoist','dhe_kJkg_moist','dTe_C','dew_margin_C',
        'h_in_kJkgdry','h_out_kJkgdry','w_in','w_out',
        'temperature_113_Dome','temperature_301_outDome'
    ]:
        out[c + '_sm'] = out[c].rolling(win, min_periods=1).mean()

    ####Venting flag.
    #Venting flag plus persistence given in vent_persist_min (in normal dome shutter operations full openning happens in 2min20sec).
    raw_vent = (out['Dome_Shutter_Actual0'] >= 50.0) & (out['Dome_Shutter_Actual1'] >= 50.0)
    persist = max(int(vent_persist_min), 1)
    #True counts as 1; we require persist consecutive Trues to consider it is actually venting.
    venting = raw_vent.rolling(persist, min_periods=persist).sum().eq(persist).fillna(False)
    out['venting'] = venting
    #Rising edge: False -> True marks the first minute the dome is fully open.
    out['vent_open_event'] = venting & (~venting.shift(1, fill_value=False))
    #CHECK
    out.to_csv('venting_flags_events.csv')
    return out


Dataset visualization of variable of interest:

In [ ]:

####Visualization.
##Marking Sun altitude at 0deg and -12deg. Shading venting/dome open period.
##The smooth_window is a rolling mean window (in minutes) to reduce noise.
##min_separation: minimum time between reported crossings.
def _find_sun_crossings(sun_series, threshold_deg, smooth_window=5, min_separation='5min'):
    s = sun_series.copy()
    if smooth_window and smooth_window > 1:
        s = s.rolling(int(smooth_window), center=True, min_periods=1).mean()
    y = s - float(threshold_deg)
    idx = s.index
    times, last_kept = [], None

    for i in range(len(s) - 1):
        y0, y1 = y.iloc[i], y.iloc[i+1]
        if not np.isfinite(y0) or not np.isfinite(y1):
            continue
        if y0 == 0:
            t_cross = idx[i]
        elif y0 * y1 < 0 or y1 == 0:
            denom = (y0 - y1)
            if denom == 0:
                continue
            f = y0 / denom
            if f < 0 or f > 1:
                continue
            dt = idx[i+1] - idx[i]
            t_cross = idx[i] + pd.Timedelta(seconds=f * dt.total_seconds())
        else:
            continue
        if last_kept is None or (t_cross - last_kept) >= pd.Timedelta(min_separation):
            times.append(t_cross)
            last_kept = t_cross
    return times

def _add_sun_vlines(axs, times_0deg, times_m12deg,
                    color_0='red', color_m12='green',
                    ls_0='-', ls_m12='--', lw=1.2, alpha=0.9):
    if not isinstance(axs, (list, tuple, np.ndarray)):
        axs = [axs]
    labeled0 = labeled12 = True
    for ax in axs:
        for t in times_m12deg:
            ax.axvline(t, color=color_m12, linestyle=ls_m12, lw=lw, alpha=alpha,
                       label='Sun −12°')
            labeled12 = True
        for t in times_0deg:
            ax.axvline(t, color=color_0, linestyle=ls_0, lw=lw, alpha=alpha,
                       label='Sun 0°')
            labeled0 = True

def _shade_venting(axs, time_index, venting_bool):
    if not isinstance(axs, (list, tuple, np.ndarray)):
        axs = [axs]
    vm = pd.Series(venting_bool, index=time_index).astype(int)
    for ax in axs:
        ax.fill_between(time_index, 0, 1, where=vm>0, color='tab:gray', alpha=0.10,
                        transform=ax.get_xaxis_transform(), label='Venting')



def _sun_state_masks(sun: pd.Series, nautical_thr=-12.0):
    day_mask      = sun > 0.0
    nautical_mask = (sun <= 0.0) & (sun > float(nautical_thr))
    night_mask    = sun <= float(nautical_thr)
    return day_mask, nautical_mask, night_mask

def _state_labels():
    return [('Day (>0°)', 'day'), ('Twilight (−12…0°)', 'twilight'), ('Night (≤−12°)', 'night')]

    
###Time Series visualization. Smoothed by the rolling mean are plotted.
def plot_time_series_with_sun_facets(feat: pd.DataFrame, sun_col='SunAltitude',
                                     width_ratios=(1, 1, 3), figsize=None):
    if sun_col not in feat.columns:
        raise KeyError(f"Column '{sun_col}' not found.")
    sun = feat[sun_col]
    day_m, twi_m, night_m = _sun_state_masks(sun)
    states = [(day_m, 'Day (>0°)'), (twi_m, 'Twilight (−12…0°)'), (night_m, 'Night (≤−12°)')]

    # 3 rows: [Temperatures], [Enthalpies], [Δh & ΔT]
    if figsize is None:
        figsize = (18, 8)
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = fig.add_gridspec(nrows=3, ncols=3, width_ratios=width_ratios)

    # Collect series once
    T_in  = feat.get('temperature_113_Dome_sm', feat.get('temperature_113_Dome'))
    T_out = feat.get('temperature_301_outDome_sm', feat.get('temperature_301_outDome'))
    Td_out = feat.get('dewPoint_301')

    h_in  = feat.get('h_in_kJkgmoist_sm', feat.get('h_in_kJkgmoist'))
    h_out = feat.get('h_out_kJkgmoist_sm', feat.get('h_out_kJkgmoist'))

    dhe = feat.get('dhe_kJkg_moist_sm', feat.get('dhe_kJkg_moist'))
    dTe = feat.get('dTe_C_sm', feat.get('dTe_C'))

    # Row 1: Temps
    axes_row1 = []
    for j, (mask, title) in enumerate(states):
        ax = fig.add_subplot(gs[0, j])
        if T_in is not None:  ax.plot(feat.index[mask], T_in[mask], label='T_in')
        if T_out is not None: ax.plot(feat.index[mask], T_out[mask], label='T_out')
        if Td_out is not None: ax.plot(feat.index[mask], Td_out[mask], label='T_dew_out', alpha=0.8)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        if j == 0: ax.set_ylabel('Temperature [°C]')
        axes_row1.append(ax)
    axes_row1[0].legend(loc='best')

    # Row 2: Enthalpies
    axes_row2 = []
    for j, (mask, _) in enumerate(states):
        ax = fig.add_subplot(gs[1, j], sharey=(axes_row2[0] if axes_row2 else None))
        if h_in is not None:  ax.plot(feat.index[mask], h_in[mask], label='h_in')
        if h_out is not None: ax.plot(feat.index[mask], h_out[mask], label='h_out')
        ax.grid(True, alpha=0.3)
        if j == 0: ax.set_ylabel('h [kJ/kg moist]')
        axes_row2.append(ax)
    axes_row2[0].legend(loc='best')

    # Row 3: Δh and ΔT
    axes_row3 = []
    for j, (mask, _) in enumerate(states):
        ax = fig.add_subplot(gs[2, j], sharey=(axes_row3[0] if axes_row3 else None))
        if dhe is not None: ax.plot(feat.index[mask], dhe[mask], label='Δh (moist)')
        if dTe is not None: ax.plot(feat.index[mask], dTe[mask], label='ΔT')
        ax.axhline(0.0, color='k', lw=0.8)
        ax.grid(True, alpha=0.3)
        if j == 0: ax.set_ylabel('Δh [kJ/kg moist], ΔT [°C]')
        ax.set_xlabel('Time (UTC)')
        axes_row3.append(ax)
    axes_row3[0].legend(loc='best')

    # Optional: venting shading per panel (only where mask is True)
    vent = feat.get('venting', None)
    if vent is not None:
        for row_axes in (axes_row1, axes_row2, axes_row3):
            for ax, (mask, _) in zip(row_axes, states):
                vm = pd.Series(vent, index=feat.index).astype(bool)
                vm = vm & mask
                if vm.any():
                    ax.fill_between(feat.index[mask], 0, 1, where=vm[mask],
                                    color='tab:gray', alpha=0.10,
                                    transform=ax.get_xaxis_transform())

    # Sun crossing lines aren’t necessary in faceted panels because each panel is a single sun state,
    # but you can add them to the Day/Twilight borders if desired.

    return fig

###Lower level diagnostics.
##Current dew point marging is 5°C.
def plot_low_level_diagnostics_with_sun_facets(feat: pd.DataFrame, sun_col='SunAltitude',
                                               dew_margin_thr_C=5.0,
                                               width_ratios=(1, 1, 3), figsize=None):
    if sun_col not in feat.columns:
        raise KeyError(f"Column '{sun_col}' not found.")
    sun = feat[sun_col]
    day_m, twi_m, night_m = _sun_state_masks(sun)
    states = [(day_m, 'Day (>0°)'), (twi_m, 'Twilight (−12…0°)'), (night_m, 'Night (≤−12°)')]

    if figsize is None:
        figsize = (18, 8)
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = fig.add_gridspec(nrows=3, ncols=3, width_ratios=width_ratios)

    dm  = feat.get('dew_margin_C_sm', feat.get('dew_margin_C'))
    w_in = feat.get('w_in_sm', feat.get('w_in'))
    w_out= feat.get('w_out_sm', feat.get('w_out'))

    # Precompute ratio
    w_avg = (feat.get('w_in_sm', feat.get('w_in'))) + (feat.get('w_out_sm', feat.get('w_out')))
    if w_avg is not None:
        w_avg = w_avg / 2.0
        cp_mavg = cp_moist_kJ_per_kg_moistK(w_avg)
    else:
        cp_mavg = None
    dT = feat.get('dTe_C_sm', feat.get('dTe_C'))
    dT_safe = dT.where(dT.abs() >= 1e-1, np.nan) if dT is not None else None
    ratio = (feat.get('dhe_kJkg_moist_sm', feat.get('dhe_kJkg_moist')) / (cp_mavg * dT_safe)
             if (cp_mavg is not None and dT_safe is not None) else None)

    # Row 1: Dew margin
    axes1 = []
    for j, (mask, title) in enumerate(states):
        ax = fig.add_subplot(gs[0, j])
        if dm is not None:
            ax.plot(feat.index[mask], dm[mask], label='Dew margin')
            ax.axhline(dew_margin_thr_C, color='r', ls='--', lw=0.9, label=f'{dew_margin_thr_C:.0f}°C threshold')
        ax.set_title(title); ax.grid(True, alpha=0.3)
        if j == 0: ax.set_ylabel('Dew margin [°C]')
        axes1.append(ax)
    axes1[0].legend(loc='best')

    # Row 2: Humidity ratios
    axes2 = []
    for j, (mask, _) in enumerate(states):
        ax = fig.add_subplot(gs[1, j], sharey=(axes2[0] if axes2 else None))
        if w_in is not None:  ax.plot(feat.index[mask], w_in[mask],  label='w_in')
        if w_out is not None: ax.plot(feat.index[mask], w_out[mask], label='w_out')
        ax.grid(True, alpha=0.3)
        if j == 0: ax.set_ylabel('w [kg/kg dry]')
        axes2.append(ax)
    axes2[0].legend(loc='best')

    # Row 3: Ratio R
    axes3 = []
    for j, (mask, _) in enumerate(states):
        ax = fig.add_subplot(gs[2, j], sharey=(axes3[0] if axes3 else None))
        if ratio is not None:
            ax.plot(feat.index[mask], ratio[mask], label='R = Δh_moist / (c_p,moist_avg · ΔT)')
            ax.axhline(1.0, color='k', lw=0.8)
            ax.set_ylim(-5, 5)
        ax.grid(True, alpha=0.3)
        if j == 0: ax.set_ylabel('Ratio R')
        ax.set_xlabel('Time (UTC)')
        axes3.append(ax)
    axes3[0].legend(loc='best')

    # Venting shading
    vent = feat.get('venting', None)
    if vent is not None:
        for row_axes in (axes1, axes2, axes3):
            for ax, (mask, _) in zip(row_axes, states):
                vm = pd.Series(vent, index=feat.index).astype(bool)
                vm = vm & mask
                if vm.any():
                    ax.fill_between(feat.index[mask], 0, 1, where=vm[mask],
                                    color='tab:gray', alpha=0.10,
                                    transform=ax.get_xaxis_transform())

    return fig


Assesing the effectiveness of the Enthalpy vs. Temperature approach:

In [ ]:
###Quadrant evaluation for enthalpy and temperature.
#Δh (moist) vs ΔT scatter colored by sun-altitude. The data points taken where venting are outlined.   
def plot_scatter_dh_dT_facets(
    feat: pd.DataFrame,
    sun_col='SunAltitude',
    mode='hex',                 # 'hex' | 'hist2d' | 'points'
    norm='log',                 # 'log' | 'linear'
    gridsize=50,                # for hex
    bins=80,                    # for hist2d
    cmap='cividis',             # colorblind-friendly
    mincnt=3,                   # suppress very sparse bins
    clip_percentiles=(1, 99),   # robust axis limits
    overlay_open=True,          # plot vent_open_event as stars
    overlay_vent=True,          # plot venting points as 'x'
    overlay_max_points=5000,    # subsample overlays if huge
    alpha_points=0.25,          # if mode='points'
    s_points=8,                 # if mode='points'
    show_colorbar=True,
    figsize=(15, 4.8)
):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.colors import LogNorm, Normalize

    if sun_col not in feat.columns:
        raise KeyError(f"Column '{sun_col}' not found.")
    # Coerce to numeric
    x = pd.to_numeric(feat.get('dTe_C'), errors='coerce')
    y = pd.to_numeric(feat.get('dhe_kJkg_moist'), errors='coerce')
    valid_all = x.notna() & y.notna()

    if not valid_all.any():
        fig, axes = plt.subplots(1, 3, figsize=figsize, constrained_layout=True)
        for ax in np.atleast_1d(axes):
            ax.text(0.5, 0.5, "No data", ha='center', va='center'); ax.axis('off')
        return fig, axes

    # Sun-state masks
    sun = feat[sun_col]
    day_mask      = sun > 0.0
    twilight_mask = (sun <= 0.0) & (sun > -12.0)
    night_mask    = sun <= -12.0
    masks = [day_mask, twilight_mask, night_mask]
    titles = ['Day (>0°)', 'Twilight (−12…0°)', 'Night (≤−12°)']

    # Robust shared limits
    xv = x[valid_all].to_numpy(float)
    yv = y[valid_all].to_numpy(float)
    p_lo, p_hi = clip_percentiles
    try:
        xlim = (np.nanpercentile(xv, p_lo), np.nanpercentile(xv, p_hi))
        ylim = (np.nanpercentile(yv, p_lo), np.nanpercentile(yv, p_hi))
    except Exception:
        xlim = (float(np.nanmin(xv)), float(np.nanmax(xv)))
        ylim = (float(np.nanmin(yv)), float(np.nanmax(yv)))

    # Norm
    use_log = (norm == 'log')
    density_norm = (LogNorm() if use_log else Normalize())

    fig, axes = plt.subplots(1, 3, figsize=figsize, sharex=True, sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    last_mappable = None  # colorbar handle

    # Overlays
    open_mask = feat.get('vent_open_event', pd.Series(False, index=feat.index)).astype(bool)
    vent_mask = feat.get('venting', pd.Series(False, index=feat.index)).astype(bool)

    for ax, state_mask, title in zip(axes, masks, titles):
        m = state_mask & valid_all
        if not m.any():
            ax.text(0.5, 0.5, "No data", ha='center', va='center')
            ax.set_title(title); ax.grid(True, alpha=0.3)
            continue

        xm = x[m].to_numpy(float); ym = y[m].to_numpy(float)

        # Clip to shared limits to prevent extreme values from compressing the view
        keep = (xm >= xlim[0]) & (xm <= xlim[1]) & (ym >= ylim[0]) & (ym <= ylim[1])
        xm = xm[keep]; ym = ym[keep]
        n_panel = int(xm.size)

        if n_panel == 0:
            ax.text(0.5, 0.5, "No data in range", ha='center', va='center')
            ax.set_title(title); ax.grid(True, alpha=0.3)
            continue

        if mode == 'hex':
            hb = ax.hexbin(xm, ym, gridsize=gridsize, mincnt=mincnt, cmap=cmap, norm=density_norm)
            last_mappable = hb
        elif mode == 'hist2d':
            h, _, _, img = ax.hist2d(xm, ym, bins=bins, cmap=cmap, norm=density_norm)
            last_mappable = img
        else:
            ax.scatter(xm, ym, s=s_points, alpha=alpha_points, c='tab:blue', edgecolors='none')

        # Overlays: openings and venting on top, with distinct markers
        # Subsample overlays to keep things readable
        if overlay_open:
            mo = (m & open_mask & valid_all)
            if mo.any():
                xs = x[mo].to_numpy(float); ys = y[mo].to_numpy(float)
                # Clip overlays to same limits
                ko = (xs >= xlim[0]) & (xs <= xlim[1]) & (ys >= ylim[0]) & (ys <= ylim[1])
                xs, ys = xs[ko], ys[ko]
                if xs.size > overlay_max_points:
                    idx = np.random.default_rng(0).choice(xs.size, size=overlay_max_points, replace=False)
                    xs, ys = xs[idx], ys[idx]
                ax.scatter(xs, ys, marker='*', s=60, facecolors='none', edgecolors='red',
                           linewidths=1.0, alpha=0.9, zorder=3, label='opening')

        # Cosmetics
        ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
        ax.set_title(f"{title} (n={n_panel})")
        ax.grid(True, alpha=0.3)

    # Shared axes/labels
    for ax in axes:
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_xlabel('ΔT = T_in − T_out [°C]')
    axes[0].set_ylabel('Δh [kJ/kg moist]')

    # Legend once (if overlays present)
    handles, labels = axes[-1].get_legend_handles_labels()
    if handles:
        axes[-1].legend(loc='best')

    # Single colorbar (density modes only)
    if show_colorbar and last_mappable is not None and mode in ('hex','hist2d'):
        cb = fig.colorbar(last_mappable, ax=axes.ravel().tolist(), fraction=0.035, pad=0.02)
        cb.set_label('counts (log scale)' if use_log else 'counts')

    return fig, axes

####Outputs for checking.
def additional_checks(feat, n=3):
    cols = ['RH_113_Dome','pressure_113_Dome_kPa',
            'RH_301_outDome','pressure_301_outDome_kPa',
            'w_in_sm','w_out_sm',
            'h_in_kJkgdry','h_in_kJkgmoist','h_in_kJkgmoist_sm','h_out_kJkgdry','h_out_kJkgmoist','h_out_kJkgmoist_sm'
            'dTe_C','dTe_C_sm']
    cols = [c for c in cols if c in feat.columns]
    print(feat[cols].head(n).to_string(float_format=lambda x: f"{x:.4f}"))

Execution of the Enthalpy vs. Temperature analysis:

In [ ]:
#Normalize data, compute equations, apply the rolling smoothing, create the venting events.
features = compute_equations(df, smooth_roll_min=3, vent_persist_min=3)

#Pressures ~74 kPa. w ~ 1e-3 values.
additional_checks(features, n=3)

#Plots with SunAltitude lines and venting shading everywhere

# Night-focused multi-panel with day/twilight collapsed into narrow columns
fig1 = plot_time_series_with_sun_facets(features, sun_col='SunAltitude', width_ratios=(1,1,3))
plt.show()

# Low-level diagnostics, same layout
fig2 = plot_low_level_diagnostics_with_sun_facets(features, sun_col='SunAltitude', width_ratios=(1,1,3))
plt.show()


# Or points with transparency
# fig3, axes3 = plot_scatter_dh_dT_facets(features, sun_col='SunAltitude', use_hexbin=False, alpha=0.25, s=8)

fig, axes = plot_scatter_dh_dT_facets(
    features,
    sun_col='SunAltitude',
    mode='points', alpha_points=0.35, s_points=6,
    overlay_open=True, overlay_vent=True,
    show_colorbar=False
)
plt.show()


Link the above results  IQ query, sampling, and processing.

In [ ]:
####Based on nightly_report_aos_eas notebook.
####Asisted bt gpt-5 and claude4.5sonet.

###Getting data ranges.
def _make_dayobs_clause(dayobs_start=None, dayobs_end=None, dayobs_list=None):
    if dayobs_list is not None:
        vals = ",".join(str(int(d)) for d in sorted(set(dayobs_list)))
        return f"AND e.day_obs IN ({vals})"
    if dayobs_start is not None and dayobs_end is not None:
        return f"AND e.day_obs >= {int(dayobs_start)} AND e.day_obs <= {int(dayobs_end)}"
    if dayobs_start is not None:
        return f"AND e.day_obs = {int(dayobs_start)}"
    raise ValueError("Provide dayobs_list or dayobs_start (and optionally dayobs_end).")

###Data query from LSSTCam ccdvisit1, visit1 and exposure.
def get_iq_range(cdb_client, dayobs_start=None, dayobs_end=None, dayobs_list=None, detectors=(191,192,195,196,199,200,203,204)):
    day_clause = _make_dayobs_clause(dayobs_start, dayobs_end, dayobs_list)
    det_list = ",".join(str(int(d)) for d in detectors)

    query = f"""
        SELECT
            e.airmass AS airmass,
            e.dimm_seeing AS dimm,
            e.altitude AS elevation,
            e.azimuth AS azimuth,
            e.exposure_id AS visit_id,
            e.physical_filter AS band,
            e.day_obs AS day_obs,
            e.exp_midpt AS time,
            e.dimm_seeing AS seeing,
            e.science_program AS science_program,
            cql.psf_sigma,
            ccdvisit1.detector AS detector,
            q.psf_sigma_median AS psf_sigma_median,
            q.psf_sigma_min AS psf_sigma_min,
            q.psf_sigma_max AS psf_sigma_max,
            q.aos_fwhm AS aos_fwhm,
            q.donut_blur_fwhm AS donut_blur_fwhm,
            e.obs_end,
            e.obs_start,
            e.seq_num AS seq
        FROM
            cdb_lsstcam.ccdvisit1_quicklook AS cql,
            cdb_lsstcam.ccdvisit1 AS ccdvisit1,
            cdb_lsstcam.visit1 AS visit1,
            cdb_lsstcam.visit1_quicklook AS q,
            cdb_lsstcam.exposure AS e
        WHERE
            ccdvisit1.detector IN ({det_list})
            AND ccdvisit1.ccdvisit_id = cql.ccdvisit_id
            AND ccdvisit1.visit_id = visit1.visit_id
            AND ccdvisit1.visit_id = q.visit_id
            AND ccdvisit1.visit_id = e.exposure_id
            AND e.img_type IN ('science')
            {day_clause}
            AND e.airmass > 0
            AND e.physical_filter <> 'none'
    """
    return cdb_client.query(query).to_pandas()

#AND e.img_type IN ('science','acq','cwfs')

##Creation of the IQ sampling choosing one source of IQ input data.
def build_iq_timeseries_range(
    cdb_table,
    resample_freq='1min',              ##<<<<<<<<<<<<<<<<<<<<<------------ SAMPLING
    iq_priority=('aos_fwhm','donut_blur_fwhm','psf_fwhm','dimm'),    ######<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< WARNING. Order of priority.
    mix_sources=False,          #Default no mixing in IQ input data. 
    chosen_source='auto'         
):
    df = cdb_table.copy()
    #Data ranges
    df['obs_start'] = pd.to_datetime(df['obs_start'], utc=True, errors='coerce')
    df = df.dropna(subset=['obs_start']).set_index('obs_start').sort_index()
    #Calculte psf_fwhm from psf_sigma_median: LSSTCam FWHM = psf_sigma_median*2.355*0.2. Plate scale 0.2"/pix, constant 2.355=2 sqrt(2 ln(2))
    df['psf_fwhm'] = 2.355 * df['psf_sigma_median'] * float(0.2)
    
    #Aggregations per obs_start across the eight detectors.
    agg = {}
    for col in ['psf_fwhm','aos_fwhm','donut_blur_fwhm','dimm','band','science_program','seq','visit_id']:
        if col in df.columns:
            #Aggregations for categorical data using just the first field per detector.
            if col in ['band','science_program','seq','visit_id']:
                agg[col] = 'first'
            #Aggregations for the numerical data using the median.    
            else:
                agg[col] = 'median'
    
    # Make 'first' deterministic by sorting within groups
    df_sorted = df.sort_values(['visit_id'], kind='stable')
    
    # Collapse to one record per visit
    df_collapsed= (
        df_sorted
        .groupby(df.index)
        .agg(agg)
        .sort_index()
    )
       
    #per_exp = df.groupby(df.index).agg(agg).sort_index()
    per_exp = df_collapsed

    #Unified IQ [arcsec] using a priority chosen (to control AOS and Dome seeing contributions to IQ)
    #Checking data is present
    available = [c for c in iq_priority if c in per_exp.columns]
    if not available:
        raise ValueError("No IQ-like columns available after aggregation. "
                         "Expected one of: " + ", ".join(iq_priority))

    #IQ according to the priority chosen. Mixed and strict approaches. The strict approach does not search for another IQ column if missing data.
    if mix_sources:
        #Mixed: first non-null by priority.
        per_exp['IQ'] = np.nan
        for c in available:
            per_exp['IQ'] = per_exp['IQ'].where(per_exp['IQ'].notna(), per_exp[c])
        per_exp['IQ_source'] = 'mixed'
    else:
        #Strict: single-source: pick one column for the entire dataset.
        if chosen_source == 'auto':
            chosen = available[0]  #First available in priority order.
        else:
            if chosen_source not in per_exp.columns:
                raise ValueError(f"Chosen IQ source '{chosen_source}' not present. "
                                 f"Available: {available}")
            chosen = chosen_source

        #Getting the IQ for the chosen column.
        per_exp['IQ'] = per_exp[chosen]
        per_exp['IQ_source'] = chosen

        #Coverage check.
        coverage = per_exp['IQ'].notna().mean()
        print(f"\nBuilding IQ time series using as strict data source '{chosen}', with a data coverage of={coverage:.1%}")
    
    #This deals with the interpolation for missing values. This leaves a lot of NANs since there are minute gaps.
    #Soft interpolation for the sampling of IQ.
    #Resample to 1 min and optionally interpolate gaps.
    #iq_1m = per_exp[['IQ']].resample(resample_freq).median()
    #Minutes to interpolate across small gaps by 'interp_limit'.
    #if allow_time_interp:
    #    iq_1m['IQ'] = iq_1m['IQ'].interpolate(method='time', limit=int(interp_limit), limit_direction='both')
    return per_exp

###Interpolation for dealing NaN values in the 1min IQ data.
##Strategy 1: median uses only data inside the minute of sampling. This is the soft interpolation.
##Strategy 2: nearest pick the single closest exposure within a maximum tolerance (tol_s seconds).
def to_minute_series_strict(per_exp, strategy='nearest', tol_s=120):
    minute_index = pd.date_range(per_exp.index.min(), per_exp.index.max(), freq='1min', tz='UTC')    ###>>>>>>>>>>>>>>>>>>. WARNING Interpolation.
    if strategy == 'median':
        iq_1m = per_exp[['IQ']].resample('1min').median()
    elif strategy == 'nearest':
        iq_1m = per_exp[['IQ']].reindex(minute_index, method='nearest', tolerance=pd.Timedelta(seconds=tol_s))
    else:
        raise ValueError("strategy must be 'median', 'nearest'")
    return iq_1m
    

##Execution
if __name__ == "__main__":
    #ConsDB client
    os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
    cdb_client = ConsDbClient('http://consdb-pq.consdb:8080/consdb')

    #IQ time range request.
    dayobs_list = (
    pd.date_range('2025-07-01', '2025-09-21', freq='D')   ###>>>>>>>-----------DATA RANGE
      .strftime('%Y%m%d').astype(int).tolist()   # use .astype(str) if you prefer strings
    )

    #Check the range requested.
    print('\nRange requested:')
    print(len(dayobs_list), dayobs_list[:3], dayobs_list[-3:])
    
    #Data query for the night(range) of interest.
    raw = get_iq_range(cdb_client, dayobs_list=dayobs_list)

    #Raw per-night exposure counts.
    #print("\nRaw exposure counts by day_obs:")
    #print(raw.groupby('day_obs').size())
    raw.to_csv('raw.csv')
    print("\nRaw dataframe:")
    print(raw.sort_values(['obs_start'], kind='stable').head(10))
    
    #IQ time series.
    per_exp = build_iq_timeseries_range(
    raw,
    resample_freq='1min',       ##<<<<<<<<<<<<<<<<<<<-------- SAMPLING
    #iq_priority=('aos_fwhm','donut_blur_fwhm','psf_fwhm','dimm'),
    iq_priority=('donut_blur_fwhm','aos_fwhm','psf_fwhm','dimm'),
    mix_sources=False,
    #chosen_source='aos_fwhm')
    chosen_source='donut_blur_fwhm')
    #chosen_source='psf_fwhm')
        
    #IQ interpolation.
    iq_1m = to_minute_series_strict(per_exp, strategy='nearest', tol_s=120)  ######<<<<<<<<<<<<SD<<<<<<<<<<<<<<<<<<<<<<<<< WARNING IQ 1min interpolation.
    print('\nThe iq_1m dataset AFTER interpolation looks like:')
    print(iq_1m.head(10))
    print(iq_1m.tail(10))
    print('\nSize of iq_1m dataset:')
    print(iq_1m.size)
    print("\nnon-NaN IQ count:", iq_1m['IQ'].count())
    iq_1m.to_csv('iq_1m.csv')
    
    ####Outputs for checking.
    #Bounds.
    print("\nTime span with on-sky data (UTC):")
    print(iq_1m.index.min(), "→", iq_1m.index.max())
    

    #Unique seq count per science_program
    #cnt = (raw.groupby(['day_obs', 'science_program'])['seq']
    #         .nunique(dropna=True)            
    #         .rename('unique_seq_count')
    #         .reset_index())

    #pd.set_option('display.max_rows', None)
    #print(cnt)
    
    #Minute bins with one field or more.
    minute_has_exposure = per_exp.resample('1min').size() > 0  ##<<<<<<<<<<<<<<<<<<<-------- SAMPLING
    print("\nMinute bins with ≥1 exposure:", minute_has_exposure.mean()*100, "%")
    print(minute_has_exposure.head(10))
    
    #Minute sampling NaN range in IQ column.
    minute_non_nan_rate = iq_1m['IQ'].notna().mean()
    print(f"\nPer-minute non-NaN rate after interpolation with tolerance (tol_s): {minute_non_nan_rate:.1%}")


Venting windows definitions, merged with IQ and defining the interpolations:

In [ ]:
####Asisted bt gpt-5 and claude4.5sonet.
#Final check for having all the data in UTC.
def ensure_utc_index(df, name='DataFrame'):
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name} must have a DatetimeIndex. Got {type(df.index)}")
    if df.index.tz is None or str(df.index.tz) != 'UTC':
        # Confirm UTC.
        df = df.tz_localize('UTC') if df.index.tz is None else df.tz_convert('UTC')
    return df.sort_index()

#Starting of the night definition as SunAltitude at -12 deg.
def time_since_minus_twelve_start(feat, sun_col='SunAltitude'):
    if sun_col not in feat.columns:
        raise KeyError(f"Column '{sun_col}' not found in features.")
    sun = feat[sun_col]
    #SunAltitude at -12 deg mask.
    mask_night = (sun <= -12.0)
    #Number of nights (block with cumsum) with a SunAltitude at -12 deg event.
    block_id = (mask_night.astype(int).diff() == 1).cumsum()
    #Outside SunAltitude=-12 deg range data is NaN.
    block_id = block_id.where(mask_night, np.nan)

    t_night_min = pd.Series(np.nan, index=feat.index)
    for bid in sorted(block_id.dropna().unique()):
        block_mask = (block_id == bid)
        if not block_mask.any():
            continue
        #Here it is the t0 event for SunAltitude=-12deg for each night block.
        t0 = feat.index[block_mask][0]
        #Time since -12deg: Current time - t0 per night block in minutes.
        t_night_min.loc[block_mask] = (feat.index[block_mask] - t0).total_seconds() / 60.0

    return t_night_min, block_id

#For building the ventind windows:
def build_vent_events(
    feat,
    iq_1m,
    tau_pre_min=20,                      
    window_pre_min=60,                   
    window_post_min=120,                 
    dew_thr_C=5.0,                       
    use_dew_gate_if_available=True,
    vent_persist_min=None,              
    vent_threshold=50.0,                  
    max_delta_min_filter=300,     
    require_same_night=True,      
    night_buffer_min=15          
):

    #Final check on UTC indices.
    feat = ensure_utc_index(feat, 'feat')
    iq_1m = ensure_utc_index(iq_1m, 'iq_1m')
    
     # Right after the UTC checks, before using opens_idx
    try:
        # Reuse your existing helper on feat to get per-timestamp night blocks
        _, _night_id = time_since_minus_twelve_start(feat, 'SunAltitude')
    except KeyError:
        _night_id = pd.Series(np.nan, index=feat.index)

    
    #Derive vent_open_event if only 'venting' exists
    if 'vent_open_event' not in feat.columns:
        if 'venting' in feat.columns:
            feat['vent_open_event'] = feat['venting'] & (~feat['venting'].shift(1, fill_value=False))
        else:
            raise KeyError("build_vent_events: need 'vent_open_event' or 'venting' to detect openings.")

    #Dew point gate for safety opening.
    dew_ok_series = pd.Series(True, index=feat.index)
    if use_dew_gate_if_available and 'dew_margin_C_sm' in feat.columns:
        dew_ok_series = feat['dew_margin_C_sm'] >= float(dew_thr_C)

    #Pick open events (rising edges) during dew-safe times.
    opens_idx = feat.index[feat['vent_open_event'].fillna(False) & dew_ok_series.reindex(feat.index, fill_value=False)]

    #Fill missing values with NaN (ensure columns exist).
    for c in ['dhe_kJkg_moist_sm', 'dTe_C_sm', 'dew_margin_C_sm', 'WindSpeed_301', 'SunAltitude', 'venting']:
        if c not in feat.columns:
            feat[c] = np.nan

    records = []
    traj_rows = []

    #Define the start of on-sky and the time to consider the pre-venting with delta_min_start via tau_pre_min.
    for t_open in opens_idx:
        #First valid IQ at/after t_open.
        # Determine search cutoff for first IQ
        cut_time = t_open + pd.Timedelta(minutes=float(max_delta_min_filter))
        
        # --- inside build_vent_events(), in the for t_open in opens_idx loop ---
        
        # 1) Bound the search window
        cut_time = t_open + pd.Timedelta(minutes=float(max_delta_min_filter))
        
        if require_same_night and (t_open in _night_id.index):
            bid = _night_id.loc[t_open]
            if pd.notna(bid):
                block_mask = (_night_id == bid)
                if block_mask.any():
                    t_night_end = _night_id.index[block_mask][-1]
                    cut_time = min(cut_time, t_night_end + pd.Timedelta(minutes=float(night_buffer_min)))
        
        # 2) First IQ within the allowed window (causal or nearest per your choice)
        iq_after = iq_1m.loc[t_open:cut_time]
        iq_valid = iq_after['IQ'].dropna()
        
        # 3) If none, skip this open event
        if iq_valid.empty:
            # Optionally record exclusion reason here
            # excluded.append({'t_open': t_open, 'reason': 'no_IQ_in_window'})
            continue
        
        # 4) Now it's safe to assign and use t_iq_start
        t_iq_start = iq_valid.index[0]
        delta_min_start = (t_iq_start - t_open).total_seconds() / 60.0
        
        # 5) Final guard (redundant given cut_time, but harmless)
        if delta_min_start > float(max_delta_min_filter):
            # excluded.append({'t_open': t_open, 'reason': 'delta_exceeds_cap'})
            continue
        
        # 6) Scenario label
        scenario = 'Pre-venting' if delta_min_start >= float(tau_pre_min) else 'On-sky'
        #Initial conditions at t_open.
        row0 = feat.loc[t_open]
        init = {
            't_open': t_open,
            't_iq_start': t_iq_start,
            'scenario': scenario,
            'delta_min_start': float(delta_min_start),
            'dh0_kJkg_moist': float(row0['dhe_kJkg_moist_sm']) if pd.notna(row0['dhe_kJkg_moist_sm']) else np.nan,
            'dT0_C': float(row0['dTe_C_sm']) if pd.notna(row0['dTe_C_sm']) else np.nan,
            'M0_C': float(row0['dew_margin_C_sm']) if pd.notna(row0['dew_margin_C_sm']) else np.nan,
            'wind0_mps': float(row0['WindSpeed_301']) if pd.notna(row0['WindSpeed_301']) else np.nan,
            'sun0_deg': float(row0['SunAltitude']) if pd.notna(row0['SunAltitude']) else np.nan,
        }
        records.append(init)

        #Build aligned blocks: [t_open - pre, t_open + post] at 1-min grid (UTC).
        t0_pre = t_open - pd.Timedelta(minutes=int(window_pre_min))
        t1_post = t_open + pd.Timedelta(minutes=int(window_post_min))
        grid = pd.date_range(t0_pre, t1_post, freq='1min', tz='UTC')

        blk = pd.DataFrame(index=grid)
        #Append environmental data from feat (left join).
        blk = blk.join(feat[['dhe_kJkg_moist_sm','dTe_C_sm','dew_margin_C_sm','WindSpeed_301','SunAltitude','venting']], how='left')
        #Append IQ by nearest-with-tolerance to handle sparse cadence and fractional seconds in IQ.
        blk['IQ'] = iq_1m['IQ'].reindex(grid, method='nearest', tolerance=pd.Timedelta(seconds=120))  ######<<<<<<<<< INTERPOLATION IN IQ.
        
        #Time since dome shutter open and since first IQ.
        blk['s_open_min'] = (blk.index - t_open).total_seconds() / 60.0     #Time since dome opened. 
        blk['s_iq_min'] = (blk.index - t_iq_start).total_seconds() / 60.0   #Time since first IQ.

        #Hard-mask IQ before first IQ (prevents any pre-IQ trend from appearing anywhere) 
        blk.loc[blk['s_iq_min'] < 0, 'IQ'] = np.nan

        #Baseline IQ during a specific period: median in 10 min before first IQ (or IQ at start).
        pre_iq_window = iq_1m.loc[max(t_open, t_iq_start - pd.Timedelta(minutes=10)):t_iq_start]
        if pre_iq_window['IQ'].notna().any():
            iq_base = float(pre_iq_window['IQ'].median())
        else:
            iq_base = float(iq_1m.loc[t_iq_start, 'IQ'])
        blk['IQ_base'] = iq_base

        #ΔIQ
        blk['dIQ'] = blk['IQ'] - iq_base

        #Recording the open event and the scenario.
        blk['event_open'] = t_open
        blk['scenario'] = scenario

        traj_rows.append(blk.reset_index().rename(columns={'index':'time'}))

    #Per-event table indexed by t_open in UTC.
    events = pd.DataFrame.from_records(records).set_index('t_open').sort_index() if records else pd.DataFrame(columns=[
        't_iq_start','scenario','delta_min_start','dh0_kJkg_moist','dT0_C','M0_C','wind0_mps','sun0_deg'
    ])
    traj = pd.concat(traj_rows, axis=0).sort_values(['event_open','time']) if traj_rows else pd.DataFrame()

    #Reporting.
    print(f"\nDetected {len(opens_idx)} dome shutter open event(s) (dew-point safe).")
    events.to_csv('events.csv')
    return events, traj


Trajectories for IQ with error bands:

In [ ]:
#Error bands definition for IQ and ΔIQ (p16/p50/p84)
def _bands_safe(df_grp):
    iq = df_grp['IQ'].to_numpy(dtype=float)
    iq = iq[~np.isnan(iq)]
    d  = df_grp['dIQ'].to_numpy(dtype=float)
    d  = d[~np.isnan(d)]

    if iq.size:
        IQ_med = float(np.median(iq)); IQ_lo = float(np.percentile(iq, 16)); IQ_hi = float(np.percentile(iq, 84))
    else:
        IQ_med = IQ_lo = IQ_hi = np.nan

    if d.size:
        dIQ_med = float(np.median(d)); dIQ_lo = float(np.percentile(d, 16)); dIQ_hi = float(np.percentile(d, 84))
    else:
        dIQ_med = dIQ_lo = dIQ_hi = np.nan

    return pd.Series({'IQ_med': IQ_med, 'IQ_lo': IQ_lo, 'IQ_hi': IQ_hi,
                      'dIQ_med': dIQ_med, 'dIQ_lo': dIQ_lo, 'dIQ_hi': dIQ_hi})


#Visualization of IQ trajectories (with bands) by scenario
def plot_iq_trajectories(traj, by='s_open_min', window_pre_min=None, window_post_min=None,
                         iq_metric_name='IQ', tau_pre_min=None,
                         show=True, return_fig=True):
     #Checking empty data.
    if traj.empty:
        print("No trajectories to plot."); 
        return None if return_fig else None
    if by not in ['s_open_min','s_iq_min']:
        raise ValueError("by must be 's_open_min' or 's_iq_min'")

    #In plot_iq_trajectories before banding:
    tmp = traj[['scenario', by, 'IQ', 'dIQ']].dropna(subset=[by]).copy()
    if by == 's_open_min':
        tmp.loc[tmp['s_open_min'] < 0, ['IQ','dIQ']] = np.nan
    tmp[by] = tmp[by].round(0)
    
    agg = tmp.groupby(['scenario', by], observed=True).apply(_bands_safe).reset_index()

    '''##Both figures. Trajectory and ΔIQ
    fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=False)
    for scen, df_s in agg.groupby('scenario', observed=True):
        axs[0].plot(df_s[by], df_s['IQ_med'], label=f'{scen}')
        axs[0].fill_between(df_s[by], df_s['IQ_lo'], df_s['IQ_hi'], alpha=0.2)
        axs[1].plot(df_s[by], df_s['dIQ_med'], label=f'{scen}')
        axs[1].fill_between(df_s[by], df_s['dIQ_lo'], df_s['dIQ_hi'], alpha=0.2)

    axs[0].set_ylabel('IQ [arcsec]'); axs[0].grid(True, alpha=0.3); axs[0].legend(loc='best')
    axs[1].axhline(0, color='k', lw=0.8)
    axs[1].set_ylabel('ΔIQ vs baseline [arcsec]')
    axs[1].set_xlabel('Minutes since ' + ('vent open' if by=='s_open_min' else 'first IQ'))
    axs[1].grid(True, alpha=0.3)
    '''
    #Single figure
    fig, axs = plt.subplots(1, 1, figsize=(10, 4))
    for scen, df_s in agg.groupby('scenario', observed=True):
        axs.plot(df_s[by], df_s['IQ_med'], label=f'{scen}')
        axs.fill_between(df_s[by], df_s['IQ_lo'], df_s['IQ_hi'], alpha=0.2)
    axs.set_ylabel('IQ [arcsec]'); axs.grid(True, alpha=0.3); axs.legend(loc='best')
    axs.set_xlabel('Minutes since ' + ('dome shutter open' if by=='s_open_min' else 'first IQ'))
    axs.grid(True, alpha=0.3)
    axs.set_ylim(0.7,2.5)

    title_bits = []
    if iq_metric_name:
        title_bits.append(f"IQ metric: {iq_metric_name}")
    if tau_pre_min is not None:
        title_bits.append(f"Venting time (vt) ={tau_pre_min} min")
    if title_bits:
        fig.suptitle(" | ".join(title_bits), y=0.98, fontsize=11)

    if show:
        plt.tight_layout(); plt.show()
    else:
        plt.tight_layout()

    return fig if return_fig else None

#Time it takes to improve IQ certain X arcsec from baseline.
#Out dataframe calculates the improvement depending on the scenario in place.
#If by == 's_open_min' a column 't_hit_s_open_min' is created.
#If by == 's_iq_min'   a column 't_hit_s_iq_min' is created.
def time_to_improvement(traj, by='s_open_min', improve_arcsec=0.2):   ######<<
    out_col = 't_hit_s_open_min' if by == 's_open_min' else 't_hit_s_iq_min'
    #Checking empty data.
    if traj.empty:
        return pd.DataFrame(columns=[out_col])

    if by not in ['s_open_min','s_iq_min']:
        raise ValueError("by must be 's_open_min' or 's_iq_min'")

    results = []
    #Group by event and scenario. observed=True avoids phantom groups for categories with no data.
    for (t_open, scen), df_e in traj.groupby(['event_open','scenario'], observed=True):
        df_e = df_e.sort_values(by)
        #Improvement: dIQ <= -abs(threshold).
        mask = (df_e['dIQ'] <= -abs(improve_arcsec))
        t_hit = float(df_e.loc[mask, by].iloc[0]) if mask.any() else np.nan
        results.append({'t_open': t_open, 'scenario': scen, out_col: t_hit})
    out = pd.DataFrame(results).set_index('t_open').sort_index()
    return out

    
#Reformatting: 't_hit_s_open_min_min' -> 't_hit_s_open_min'.
def _normalize_t_hit_columns(df):
    rename_map = {}
    for c in df.columns:
        if c.endswith('_min_min'):
            rename_map[c] = c[:-4]  #Drop the extra '_min'.
    return df.rename(columns=rename_map) if rename_map else df


#Reformatting: Adds minutes since SunAltitude at -12 deg) per night, and a night_id to the events table.
def attach_time_of_night_to_events(events, feat, max_tolerance='5min'):
    feat = ensure_utc_index(feat, 'feat')
    t_night_min, night_id = time_since_minus_twelve_start(feat, 'SunAltitude')
    tno = t_night_min.reindex(events.index, method='nearest', tolerance=pd.Timedelta(max_tolerance))
    nid = night_id.reindex(events.index, method='nearest', tolerance=pd.Timedelta(max_tolerance))
    out = events.copy()
    out['t_night_min_at_open'] = tno
    out['night_id'] = nid
    return out


# Per-night summaries
def summarize_by_night(events, t_open_hit, t_iq_hit):
    ev = events.copy()

    # Normalize accidental names if needed
    t_open_hit = _normalize_t_hit_columns(t_open_hit)
    t_iq_hit   = _normalize_t_hit_columns(t_iq_hit)

    cols_open = [c for c in t_open_hit.columns if c.startswith('t_hit_')]
    cols_iq   = [c for c in t_iq_hit.columns   if c.startswith('t_hit_')]

    # Join only missing t_hit columns to avoid overlap errors
    miss_open = [c for c in cols_open if c not in ev.columns]
    if miss_open:
        ev = ev.join(t_open_hit[miss_open], how='left')
    miss_iq = [c for c in cols_iq if c not in ev.columns]
    if miss_iq:
        ev = ev.join(t_iq_hit[miss_iq], how='left')

    # Guarantee expected columns exist
    for col in ['t_hit_s_open_min', 't_hit_s_iq_min']:
        if col not in ev.columns:
            ev[col] = np.nan

    if 'night_id' not in ev.columns:
        print("summarize_by_night: 'night_id' not found in events. Call attach_time_of_night_to_events first.")
        return pd.DataFrame()

    def scen_prop(g):
        vc = g['scenario'].value_counts(normalize=True)
        return pd.Series({
            'n_events': len(g),
            'p_pre_venting': float(vc.get('Pre-venting', 0.0)),
            'p_on_sky_venting': float(vc.get('On-sky', 0.0)),
            'delta_min_start_med': float(np.nanmedian(g['delta_min_start'])),
            't_hit_s_open_min_med': float(np.nanmedian(g['t_hit_s_open_min'])),
            't_hit_s_iq_min_med': float(np.nanmedian(g['t_hit_s_iq_min'])),
            'dh0_med': float(np.nanmedian(g['dh0_kJkg_moist'])),
            'dT0_med': float(np.nanmedian(g['dT0_C'])),
            'M0_med': float(np.nanmedian(g['M0_C'])),
            'wind0_med': float(np.nanmedian(g['wind0_mps'])),
        })

    night_summary = ev.groupby('night_id', dropna=False).apply(scen_prop).reset_index()
    return night_summary

# tau value
def suggest_tau_from_cdf(events, night_window=(0, 120), quantile=0.80, tno_col='t_night_min_at_open'):
    if 'delta_min_start' not in events.columns:
        raise KeyError("events must have delta_min_start. Build events first.")
    if tno_col not in events.columns:
        print(f"Warning: {tno_col} not in events; using all events.")
        sub = pd.to_numeric(events['delta_min_start'], errors='coerce').dropna()
    else:
        mask = (events[tno_col] >= night_window[0]) & (events[tno_col] < night_window[1])
        sub = pd.to_numeric(events.loc[mask, 'delta_min_start'], errors='coerce').dropna()
    if sub.empty:
        return np.nan
    return float(np.nanpercentile(sub, quantile*100.0))

def plot_tau_hist_cdf(events, night_window=(0, 120), tno_col='t_night_min_at_open', tau_lines=(10,20,30), binsize=1.0):
    if 'delta_min_start' not in events.columns:
        raise KeyError("events must have delta_min_start. Build events first.")
    if tno_col in events.columns:
        mask = (events[tno_col] >= night_window[0]) & (events[tno_col] < night_window[1])
        sub = pd.to_numeric(events.loc[mask, 'delta_min_start'], errors='coerce').dropna()
    else:
        sub = pd.to_numeric(events['delta_min_start'], errors='coerce').dropna()
    if sub.empty:
        print("No events to plot.")
        return
    vmax = max(5.0, float(np.nanpercentile(sub, 99)))
    edges = np.arange(0.0, vmax + binsize, binsize)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axs[0].hist(sub, bins=edges, color='tab:blue', alpha=0.75, edgecolor='white')
    axs[0].set_xlabel('Δstart = t_iq_start − t_open (min)'); axs[0].set_ylabel('Count'); axs[0].grid(True, alpha=0.3)
    s = np.sort(sub.values); cdf = np.arange(1, len(s)+1)/len(s)
    axs[1].plot(s, cdf, color='tab:purple', lw=2); axs[1].set_ylim(0, 1.01)
    axs[1].set_xlabel('Δstart (min)'); axs[1].set_ylabel('CDF'); axs[1].grid(True, alpha=0.3)
    for tau in (tau_lines or []):
        axs[0].axvline(tau, color='red', ls='--', lw=1.2)
        axs[1].axvline(tau, color='red', ls='--', lw=1.2)
    plt.show()

def reclassify_for_tau(events, traj, tau):
    ev = events.copy()
    ev['scenario'] = np.where(pd.to_numeric(ev['delta_min_start'], errors='coerce') >= float(tau),
                              'Pre-venting', 'On-sky')
    if traj is None or traj.empty:
        return ev, traj
    m = ev['scenario'].to_dict()  # key=t_open, value=scenario
    tr = traj.copy()
    tr['scenario'] = tr['event_open'].map(m)
    return ev, tr


def _median_diq_at_minutes(traj_tau, by='s_open_min', minutes=(30, 60)):
    out = {}
    if traj_tau.empty:
        for m in minutes: out[f'dIQ_med_diff_{int(m)}'] = np.nan
        return out
    df = traj_tau.copy()
    df = df[[by, 'scenario', 'dIQ']].dropna(subset=[by]).copy()
    df = df[df[by] >= 0]
    df[by] = df[by].round(0)
    grp = df.groupby(['scenario', by])['dIQ'].median().unstack(0)  # columns by scenario
    for m in minutes:
        if m in grp.index and {'Pre-venting','On-sky'}.issubset(grp.columns):
            out[f'dIQ_med_diff_{int(m)}'] = float(grp.loc[m, 'Pre-venting'] - grp.loc[m, 'On-sky'])
        else:
            out[f'dIQ_med_diff_{int(m)}'] = np.nan
    return out

def _median_t_hit_by_scenario(traj_tau, improve_arcsec=0.1):
    if traj_tau.empty:
        return {'t_hit_open_pre_med': np.nan, 't_hit_open_on_med': np.nan,
                't_hit_iq_pre_med': np.nan,   't_hit_iq_on_med': np.nan}
    t_open_hit = time_to_improvement(traj_tau, by='s_open_min', improve_arcsec=improve_arcsec)
    t_iq_hit   = time_to_improvement(traj_tau, by='s_iq_min',   improve_arcsec=improve_arcsec)
    res = {}
    for scen, lab in [('Pre-venting','Pre-venting'), ('On-sky','On-sky')]:
        res[f't_hit_open_{lab}_med'] = float(np.nanmedian(t_open_hit.loc[t_open_hit['scenario']==scen, 't_hit_s_open_min'])) if not t_open_hit.empty else np.nan
        res[f't_hit_iq_{lab}_med']   = float(np.nanmedian(t_iq_hit.loc[t_iq_hit['scenario']==scen, 't_hit_s_iq_min']))     if not t_iq_hit.empty else np.nan
    return res

def _counts_by_scenario(events_tau):
    vc = events_tau['scenario'].value_counts()
    return {
        'n_pre': int(vc.get('Pre-venting', 0)),
        'n_on':  int(vc.get('On-sky', 0)),
        'n_all': int(len(events_tau)),
    }

def sweep_tau(events, traj, tau_grid, eval_minutes=(30, 60), improve_arcsec=0.1, min_per_scen=5):
    rows = []
    for tau in tau_grid:
        ev_tau, tr_tau = reclassify_for_tau(events, traj, tau)
        sep = _median_diq_at_minutes(tr_tau, by='s_open_min', minutes=eval_minutes)
        thit = _median_t_hit_by_scenario(tr_tau, improve_arcsec=improve_arcsec)
        cnt = _counts_by_scenario(ev_tau)
        # Simple score: prioritize larger separation at 30/60 (more negative = larger improvement for Pre-venting if negative),
        # but keep scenarios sufficiently populated.
        penalty = 0.0
        if cnt['n_pre'] < min_per_scen or cnt['n_on'] < min_per_scen:
            penalty = 1.0  # penalize low counts
        score = 0.0
        for m in eval_minutes:
            v = sep.get(f'dIQ_med_diff_{int(m)}', np.nan)
            # We want Pre-venting to have more negative dIQ than On-sky (i.e., v is more negative).
            # Convert to positive "goodness": -(v). Ignore NaN.
            if np.isfinite(v): score += -v
        score -= penalty  # apply penalty
        row = {'tau': float(tau), 'score': float(score)}
        row.update(cnt); row.update(sep); row.update(thit)
        rows.append(row)
    return pd.DataFrame(rows).sort_values('tau')

def plot_tau_sweep(df_scores, eval_minutes=(30,60)):
    if df_scores.empty:
        print("No sweep results to plot.")
        return
    fig, axs = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    ax = axs[0,0]
    ax.plot(df_scores['tau'], df_scores['score'], marker='o')
    ax.set_title('Score vs tau'); ax.set_xlabel('tau_pre_min [min]'); ax.grid(True, alpha=0.3)

    ax = axs[0,1]
    for m in eval_minutes:
        col = f'dIQ_med_diff_{int(m)}'
        if col in df_scores.columns:
            ax.plot(df_scores['tau'], df_scores[col], marker='o', label=f'Δ(dIQ_med) @ {m} min')
    ax.axhline(0, color='k', lw=0.8); ax.grid(True, alpha=0.3); ax.legend(loc='best')
    ax.set_title('Median Δ(dIQ) (Pre − On)')

    ax = axs[1,0]
    ax.plot(df_scores['tau'], df_scores['n_pre'], label='Pre-venting', color='tab:orange')
    ax.plot(df_scores['tau'], df_scores['n_on'],  label='On-sky', color='tab:blue')
    ax.set_title('Counts by scenario vs tau'); ax.set_xlabel('Ventilation time [min]'); ax.set_ylabel('N Dome shutter openings')
    ax.grid(True, alpha=0.3); ax.legend(loc='best')

    ax = axs[1,1]
    for c, lab in [('t_hit_open_pre_med','open pre'), ('t_hit_open_on_med','open on'),
                   ('t_hit_iq_pre_med','iq pre'),   ('t_hit_iq_on_med','iq on')]:
        if c in df_scores.columns:
            ax.plot(df_scores['tau'], df_scores[c], marker='o', label=lab)
    ax.set_title('Median time-to-improvement vs tau'); ax.set_xlabel('tau_pre_min [min]')
    ax.grid(True, alpha=0.3); ax.legend(loc='best')
    plt.show()

def plot_tau_grid_trajectories(
    traj,
    tau_grid,
    window_post_min=120,
    by='s_open_min',
    include_ci=True,                 # draw p16–p84 bands
    colors=None,                     # optional color override: {'Pre-venting': ..., 'On-sky + venting': ...}
    figsize=None,                    # default scales with number of taus
    show=True
):
    if traj.empty:
        print("No trajectories.")
        return None, None

    if 'delta_min_start' not in traj.columns:
        raise ValueError("traj missing 'delta_min_start'; merge it from events (left_on='event_open', right_index=True).")

    if by not in ('s_open_min', 's_iq_min'):
        raise ValueError("by must be 's_open_min' or 's_iq_min'")

    # Colors
    colors = colors or {'Pre-venting': 'tab:orange', 'On-sky': 'tab:blue'}
    scen_order = ['Pre-venting', 'On-sky']

    n = len(tau_grid)
    if figsize is None:
        figsize = (max(20, 3.6*n), 4)

    fig, axes = plt.subplots(1, n, figsize=figsize, sharex=True, sharey=True, constrained_layout=True)
    if n == 1:
        axes = np.array([axes])

    # Common x-range limits
    xlo, xhi = 0.0, float(window_post_min)

    for j, tau in enumerate(tau_grid):
        ax = axes[j]

        # Reclassify per tau at the EVENT level, then map to traj rows
        tr = traj.copy()
        # Ensure numeric
        dms = pd.to_numeric(tr['delta_min_start'], errors='coerce')
        tr['scenario'] = np.where(dms >= float(tau), 'Pre-venting', 'On-sky')

        # Count events per scenario (not rows)
        ev_tau = tr[['event_open', 'scenario']].drop_duplicates()
        vc = ev_tau['scenario'].value_counts()
        n_pre = int(vc.get('Pre-venting', 0))
        n_on  = int(vc.get('On-sky', 0))

        # Prepare data by time
        tmp = tr[[by, 'scenario', 'IQ']].dropna(subset=[by]).copy()
        tmp = tmp[tmp[by].between(0, window_post_min)]
        tmp[by] = tmp[by].round(0)

        # Aggregate: median and 16/84 percentiles
        if tmp.empty:
            ax.set_title(f"tau={int(tau)} (no data)")
            ax.grid(True, alpha=0.3)
            continue

        agg = (
            tmp.groupby(['scenario', by], observed=True)['IQ']
               .agg(IQ_med='median',
                    IQ_lo=lambda x: np.nanpercentile(x, 16),
                    IQ_hi=lambda x: np.nanpercentile(x, 84))
               .reset_index()
        )

        # Plot both scenarios overlapped
        any_plotted = False
        for scen in scen_order:
            g = agg[agg['scenario'] == scen]
            if g.empty:
                continue
            any_plotted = True
            c = colors.get(scen, None)
            ax.plot(g[by], g['IQ_med'], label=scen, color=c, lw=2)
            if include_ci:
                ax.fill_between(g[by], g['IQ_lo'], g['IQ_hi'], color=c, alpha=0.20, linewidth=0)

        ax.set_xlim(xlo, xhi)
        if by == 's_open_min':
            ax.set_xlabel('Minutes since vent open')
        else:
            ax.set_xlabel('Minutes since first IQ')

        # Title shows tau and event counts in each scenario
        ax.set_title(f"tau={int(tau)} | Pre={n_pre}, On={n_on}")
        ax.grid(True, alpha=0.3)

        # Only add legend on the first subplot that actually plotted data
        if j == 0 and any_plotted:
            ax.legend(loc='best')

    # Y label on the leftmost subplot
    axes[0].set_ylabel('IQ [arcsec]')
    if show:
        plt.show()
    return fig, axes

In [ ]:
#Main function.
def run_venting(feat, iq_1m, tau_pre_min=20, window_pre_min=60, window_post_min=120,
                dew_thr_C=5.0, iq_metric_name='aos_fwhm'):
    #Checking common UTC.
    feat = ensure_utc_index(feat, 'feat')
    iq_1m = ensure_utc_index(iq_1m, 'iq_1m')
    print("Features index tz:", feat.index.tz, " | iq_1m index tz:", iq_1m.index.tz)

    #Time since Sun at -12deg. The column is stored as t_night_min.
    try:
        feat['t_night_min'], _ = time_since_minus_twelve_start(feat, 'SunAltitude')
    except KeyError:
        feat['t_night_min'] = np.nan

    #Build events and trajectories.
    events, traj = build_vent_events(
        feat, iq_1m,
        tau_pre_min=tau_pre_min,
        window_pre_min=window_pre_min,
        window_post_min=window_post_min,
        dew_thr_C=dew_thr_C,
        use_dew_gate_if_available=True
    )

    #Checking empty data (closed nights).
    if events.empty:
        print("No dome shutter open events found under dew-safe conditions.")
        return events, traj, pd.DataFrame(), pd.DataFrame()

    #Adds minutes since SunAltitude at -12 deg) per night, and a night_id to the events table summaries.
    events_with_tno = attach_time_of_night_to_events(events, feat, max_tolerance='2min')

    print("\nEvent counts by scenario:")
    print(events_with_tno['scenario'].value_counts(dropna=True))

    print("\nEvent initial conditions (head):")
    pd.set_option('display.max_rows', None)
    #print(events_with_tno[['scenario','delta_min_start','dh0_kJkg_moist','dT0_C','M0_C','wind0_mps','sun0_deg']].head(25))  ######<<<<<<<<<<<<< DEBUGGING.
    print(events_with_tno[['scenario','delta_min_start','dh0_kJkg_moist','dT0_C','M0_C','wind0_mps','sun0_deg']])  ######<<<<<<<<<<<<< DEBUGGING.
    
    #Plot trajectories vs time since vent open and since first IQ (with labels).
    plot_iq_trajectories(traj, by='s_open_min',
                         window_pre_min=window_pre_min, window_post_min=window_post_min,
                         iq_metric_name=iq_metric_name, tau_pre_min=tau_pre_min)

    plot_iq_trajectories(traj, by='s_iq_min',
                         window_pre_min=window_pre_min, window_post_min=window_post_min,
                         iq_metric_name=iq_metric_name, tau_pre_min=tau_pre_min)


    #Time-to-improvement (0.1").
    t_open_hit = _normalize_t_hit_columns(time_to_improvement(traj, by='s_open_min', improve_arcsec=0.1))
    t_iq_hit   = _normalize_t_hit_columns(time_to_improvement(traj, by='s_iq_min',   improve_arcsec=0.1))

    #Keep only the numeric columns.
    cols_open = [c for c in t_open_hit.columns if c.startswith('t_hit_')]
    cols_iq   = [c for c in t_iq_hit.columns   if c.startswith('t_hit_')]

    summary = (
        events_with_tno
        .join(t_open_hit[cols_open], how='left')
        .join(t_iq_hit[cols_iq],     how='left'))

    #Guarantee expected columns exist.
    for col in ['t_hit_s_open_min','t_hit_s_iq_min']:
        if col not in summary.columns:
            summary[col] = np.nan

    print("\nMedian time-to-improvement (0.1\") by scenario:")
    med = summary.groupby('scenario')[['t_hit_s_open_min','t_hit_s_iq_min']].median()
    print(med)

    #Per-night summaries (informative) — pass events_with_tno (not summary) to avoid re-joining overlaps
    night_summary = summarize_by_night(events_with_tno, t_open_hit, t_iq_hit)
    if not night_summary.empty:
        print("\nPer-night summary:")
        print(night_summary)

    #Distribution of IQ at acquisition (s_iq_min=0) per scenario (optional quick view)
    acq = traj[np.isclose(traj['s_iq_min'], 0.0, atol=0.5)].copy()
    if not acq.empty:
        print("\nIQ at first acquisition (arcsec) by scenario (median [p16, p84]):")
        for scen, df_s in acq.groupby('scenario', observed=True):
            vals = df_s['IQ'].dropna().values
            if vals.size:
                print(f"  {scen}: {np.nanmedian(vals):.3f} "
                      f"[{np.nanpercentile(vals,16):.3f}, {np.nanpercentile(vals,84):.3f}]")
    else:
        print("\nNo IQ samples exactly at first acquisition; skip quick summary.")

    return events_with_tno, traj, summary, night_summary


We are interested in the impact on image quality of the term called "donut_blur_fwhm":

In [ ]:
events_with_tno, traj, summary, night_summary = run_venting(
    features, iq_1m,
    tau_pre_min=30,
    window_pre_min=30,
    window_post_min=200,
    dew_thr_C=5.0,
    #iq_metric_name='psf_fwhm'
    iq_metric_name='donut_blur_fwhm'
    #iq_metric_name='aos_fwhm'
)


In [ ]:
tau0 = suggest_tau_from_cdf(events_with_tno, night_window=(0,120), quantile=0.80)
tau0

In [ ]:
tau_grid = list(range(max(0, int(tau0)-20), int(tau0)+21, 5))

In [ ]:
df_scores = sweep_tau(events_with_tno, traj, tau_grid, eval_minutes=(30, 60), improve_arcsec=0.1, min_per_scen=5)
plot_tau_sweep(df_scores, eval_minutes=(30, 60))

In [ ]:
traj = traj.merge(events_with_tno[['delta_min_start']], left_on='event_open', right_index=True, how='left')


In [ ]:
plot_tau_grid_trajectories(traj, tau_grid=tau_grid[:12], window_post_min=120, by='s_open_min')

Getting the best tau beyond data balance:

In [ ]:
def per_event_iq_at_minutes(traj, minutes=(30, 60), by='s_open_min'):
    if traj.empty:
        return pd.DataFrame()
    t = traj.copy()
    keep = [by, 'event_open', 'IQ']
    t = t.dropna(subset=[by, 'event_open', 'IQ'])
    t[by] = pd.to_numeric(t[by], errors='coerce').round(0)
    g = t.groupby(['event_open', by], observed=True)['IQ'].median().reset_index()
    rows = []
    for ev, sub in g.groupby('event_open', observed=True):
        row = {'event_open': ev}
        for m in minutes:
            v = sub.loc[sub[by] == float(m), 'IQ']
            row[f'IQ_{int(m)}m'] = float(v.iloc[0]) if not v.empty else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index('event_open').sort_index()


def outcome_curve_by_tau(events, traj, tau_grid, minute=60, by='s_open_min',
                         min_events=8, n_boot=2000, seed=0):
    # Build per-event IQ@minute table once
    iq_ev = per_event_iq_at_minutes(traj, minutes=(minute,), by=by)
    ev = events[['delta_min_start']].join(iq_ev, how='left')
    ycol = f'IQ_{int(minute)}m'
    rng = np.random.default_rng(seed)

    out = []
    for tau in tau_grid:
        sub = ev.loc[pd.to_numeric(ev['delta_min_start'], errors='coerce') >= float(tau), ycol].dropna()
        n = int(sub.size)
        if n < min_events:
            out.append({'tau': float(tau), 'n': n, 'median': np.nan, 'lo95': np.nan, 'hi95': np.nan})
            continue
        med = float(np.nanmedian(sub))
        # Bootstrap 95% CI for the median
        if n_boot and n > 1:
            idx = rng.integers(0, n, size=(n_boot, n))
            boots = np.nanmedian(sub.to_numpy()[idx], axis=1)
            lo95, hi95 = float(np.nanpercentile(boots, 2.5)), float(np.nanpercentile(boots, 97.5))
        else:
            lo95 = hi95 = np.nan
        out.append({'tau': float(tau), 'n': n, 'median': med, 'lo95': lo95, 'hi95': hi95})
    return pd.DataFrame(out).sort_values('tau')


def plot_outcome_curve(df_curve, minute=60, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,4), constrained_layout=True)
    else:
        fig = ax.figure
    ax.plot(df_curve['tau'], df_curve['median'], color='tab:green', marker='o') #, label=f'Median IQ@{minute}m')
    if {'lo95','hi95'}.issubset(df_curve.columns):
        ax.fill_between(df_curve['tau'], df_curve['lo95'], df_curve['hi95'], color='tab:green', alpha=0.15, linewidth=0)
    ax.set_xlabel('Ventilation time [min]')
    ax.set_ylabel(f'Median IQ at {minute} min [arcsec]')
    ax.grid(True, alpha=0.3)
    if title:
        ax.set_title(title)
    # Suggested choice: smallest tau within 1% of global min median
    y = df_curve['median'].to_numpy(float)
    x = df_curve['tau'].to_numpy(float)
    if np.isfinite(y).any():
        y_min = np.nanmin(y)
        within = np.where(y <= 1.01*y_min)[0]
        if within.size:
            tau_star = x[within[0]]
            ax.axvline(tau_star, color='tab:red', ls='--', lw=1.2, label=f'Minimun value at {int(tau_star)} min')
    ax.legend(loc='best')
    return fig, ax



In [ ]:
tau_grid = list(range(0, 61, 3))  # 0..60 by 5
curve = outcome_curve_by_tau(events_with_tno, traj, tau_grid, minute=600, by='s_open_min',
                             min_events=8, n_boot=2000)
fig, ax = plot_outcome_curve(curve, minute=600, title=' ')

In [ ]:
d = pd.to_numeric(events_with_tno['delta_min_start'], errors='coerce').dropna()
print("Δstart range:", float(d.min()), float(d.max()))
p5, p95 = np.nanpercentile(d, [5, 95])
print("Δstart central support [p5,p95]:", p5, p95)

# Restrict tau grid to central support (or at least [min,max])
tau_grid_in = [t for t in tau_grid if (t >= p5 and t <= p95)]

from patsy import dmatrix

def fit_quantile_spline_with_knots(events, traj, minute=60, q=0.5, knot_perc=(20,40,60,80)):
    # Build outcome per event
    iq_ev = per_event_iq_at_minutes(traj, minutes=(minute,), by='s_open_min')
    df = events.join(iq_ev, how='left').dropna(subset=[f'IQ_{int(minute)}m','delta_min_start']).copy()
    df['delta_min_start'] = pd.to_numeric(df['delta_min_start'], errors='coerce')

    # Interior knots at requested percentiles
    knots = np.nanpercentile(df['delta_min_start'], list(knot_perc))
    # Natural cubic regression spline via cr(...) or bs(..., knots=...)
    # Patsy cr() is a natural cubic regression spline:
    X = dmatrix(f"cr(delta_min_start, knots={list(knots)}, constraints='center')",
                df, return_type='dataframe')
    y = pd.to_numeric(df[f'IQ_{int(minute)}m'], errors='coerce')

    # Drop non-finite
    m = y.notna().to_numpy() & np.isfinite(X.to_numpy(float)).all(axis=1)
    yv, Xv = y.to_numpy(float)[m], X.to_numpy(float)[m]

    import statsmodels.api as sm
    res = sm.QuantReg(yv, Xv).fit(q=q)

    # Aligned predictor builder
    exog_names = list(X.columns)
    def build_pred(new_df):
        Xn = dmatrix(f"cr(delta_min_start, knots={list(knots)}, constraints='center')",
                     new_df, return_type='dataframe')
        Xn = Xn.reindex(columns=exog_names, fill_value=0.0)
        return Xn.to_numpy(float)
    return res, build_pred


def plot_binned_medians(events, traj, minute=60, bins=10):
    iq_ev = per_event_iq_at_minutes(traj, minutes=(minute,), by='s_open_min')
    df = events.join(iq_ev, how='left').copy()
    df['delta_min_start'] = pd.to_numeric(df['delta_min_start'], errors='coerce')
    df = df.dropna(subset=['delta_min_start', f'IQ_{int(minute)}m'])
    df['bin'] = pd.qcut(df['delta_min_start'], q=bins, duplicates='drop')
    g = df.groupby('bin')[f'IQ_{int(minute)}m'].median()
    x = [b.mid for b in g.index.categories]
    y = g.values
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4)); plt.plot(x, y, marker='o')
    plt.xlabel('Δstart (bin midpoints)'); plt.ylabel(f'Median IQ@{minute}m [arcsec]')
    plt.grid(True, alpha=0.3); plt.show()

Now that the venting model exists we can try to connect the results to the weather conditions when the ventilation process starts:

In [ ]:
def report_open_timing_histograms(
    events: pd.DataFrame,
    feat: pd.DataFrame,
    # Columns
    sun_col: str = 'SunAltitude',
    wind_speed_col: str = 'WindSpeed_301',
    wind_dir_col: str = 'WindDirection_301',
    dome_az_col: str = 'Dome_Azimuth_Actual',
    delta_min_start_col: str = 'delta_min_start',
    # Binning configuration
    bin_size_min: float = 5.0,
    bin_size_deg: float = 1.0,
    bin_size_dT: float = 0.2,
    bin_size_dH: float = 1.0,
    bin_size_wind: float = 0.5,
    bin_size_dir_deg: float = 10.0,
    altitudes_deg = (-6.0, -18.0),
    # Matching tolerances
    sun_match_tolerance: str = '10min',
    env_match_tolerance: str = '10min',
    wind_match_tolerance: str = '10min',
    dome_az_match_tolerance: str = '10min',
    # ΔT and Δh candidates
    dT_col_candidates = ('dTe_C_sm', 'dTe_C'),
    dH_col_candidates = ('dhe_kJkg_moist_sm', 'dhe_kJkg'),
    # Output controls
    show: bool = True,
    return_figs: bool = False,
):

    def _ensure_utc_index(df, name):
        if not isinstance(df.index, pd.DatetimeIndex):
            raise TypeError(f"{name} must have a DatetimeIndex.")
        return df.tz_localize('UTC') if df.index.tz is None else df.tz_convert('UTC')

    def _to_float_series(s):
        return pd.to_numeric(s, errors='coerce')

    def _crossings_down(s: pd.Series, thr: float) -> pd.DatetimeIndex:
        """
        Downward crossings of s across thr (prev > thr and curr <= thr), with linear interpolation.
        """
        s = _to_float_series(s)
        idx = s.index
        vals = s.to_numpy(dtype=float)
        out = []
        for i in range(1, len(s)):
            v0, v1 = vals[i-1], vals[i]
            if not (np.isfinite(v0) and np.isfinite(v1)):
                continue
            if v0 > thr and v1 <= thr:
                t0, t1 = idx[i-1], idx[i]
                alpha = 1.0 if v1 == v0 else (thr - v0) / (v1 - v0)
                alpha = float(np.clip(alpha, 0.0, 1.0))
                t_cross = t0 + (t1 - t0) * alpha
                out.append(pd.Timestamp(t_cross).tz_convert(idx.tz) if idx.tz is not None else pd.Timestamp(t_cross))
        return pd.DatetimeIndex(out, tz=idx.tz).sort_values()

    def _first_on_or_after(times: pd.DatetimeIndex, t: pd.Timestamp):
        if len(times) == 0:
            return pd.NaT
        pos = times.searchsorted(t, side='left')
        return times[pos] if pos < len(times) else pd.NaT

    def _last_on_or_before(times: pd.DatetimeIndex, t: pd.Timestamp):
        if len(times) == 0:
            return pd.NaT
        pos = times.searchsorted(t, side='right') - 1
        return times[pos] if pos >= 0 else pd.NaT

    def _make_edges(v: np.ndarray, binsize: float):
        v = np.asarray(v, dtype=float)
        v = v[np.isfinite(v)]
        if v.size == 0:
            return np.linspace(-binsize, binsize, 3)
        vmin, vmax = np.nanpercentile(v, [1, 99])
        if not np.isfinite(vmin) or not np.isfinite(vmax):
            vmin, vmax = np.nanmin(v), np.nanmax(v)
        if vmin == vmax:
            vmin -= binsize
            vmax += binsize
        left = np.floor(vmin / binsize) * binsize
        right = np.ceil(vmax / binsize) * binsize
        nb = max(5, int(np.ceil((right - left) / binsize)))
        edges = left + np.arange(nb + 1) * binsize
        return edges

    def _hist_summary(vals: pd.Series, binsize: float):
        v = _to_float_series(vals).to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        out = {'n': int(v.size)}
        if v.size == 0:
            out.update({
                'median': np.nan, 'p16': np.nan, 'p84': np.nan,
                'mode': np.nan, 'mode_bin': (np.nan, np.nan),
                'counts': np.array([], dtype=int), 'bin_edges': np.array([], dtype=float),
            })
            return out
        edges = _make_edges(v, binsize)
        counts, bin_edges = np.histogram(v, bins=edges)
        mode_idx = int(np.argmax(counts)) if counts.size else 0
        left, right = bin_edges[mode_idx], bin_edges[mode_idx + 1]
        out.update({
            'median': float(np.nanmedian(v)),
            'p16': float(np.nanpercentile(v, 16)),
            'p84': float(np.nanpercentile(v, 84)),
            'mode': float(0.5 * (left + right)),
            'mode_bin': (float(left), float(right)),
            'counts': counts,
            'bin_edges': bin_edges
        })
        return out

    def _hist_summary_direction(vals_deg: pd.Series, bin_size_deg: float):
        v = _to_float_series(vals_deg).to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        out = {'n': int(v.size)}
        if v.size == 0:
            out.update({
                'circ_mean_deg': np.nan, 'circ_std_deg': np.nan,
                'mode_deg': np.nan, 'mode_bin': (np.nan, np.nan),
                'counts': np.array([], dtype=int), 'bin_edges': np.array([], dtype=float),
            })
            return out
        v = np.mod(v, 360.0)
        # Histogram edges from 0 to 360
        edges = np.arange(0.0, 360.0 + bin_size_deg, bin_size_deg, dtype=float)
        if edges[-1] > 360.0 + 1e-9:
            edges[-1] = 360.0
        counts, bin_edges = np.histogram(v, bins=edges, range=(0.0, 360.0))
        mode_idx = int(np.argmax(counts)) if counts.size else 0
        left, right = bin_edges[mode_idx], bin_edges[mode_idx + 1]
        # Circular mean/std
        theta = np.deg2rad(v)
        C = np.nanmean(np.cos(theta))
        S = np.nanmean(np.sin(theta))
        mean_angle = np.arctan2(S, C)
        if mean_angle < 0:
            mean_angle += 2 * np.pi
        Rbar = float(np.hypot(C, S))
        Rbar = max(Rbar, 1e-12)  # guard log(0)
        circ_std_rad = np.sqrt(-2.0 * np.log(Rbar))
        out.update({
            'circ_mean_deg': float(np.rad2deg(mean_angle)),
            'circ_std_deg': float(np.rad2deg(circ_std_rad)),
            'mode_deg': float(0.5 * (left + right)),
            'mode_bin': (float(left), float(right)),
            'counts': counts,
            'bin_edges': bin_edges
        })
        return out

    def _rename_with_unit(summary, unit_suffix):
        # Convert generic keys to unit-specific (e.g., _min, _deg, _C, _kJkg, _mps)
        if summary['n'] == 0:
            return {**summary,
                    f'median_{unit_suffix}': np.nan,
                    f'p16_{unit_suffix}': np.nan,
                    f'p84_{unit_suffix}': np.nan,
                    f'mode_{unit_suffix}': np.nan}
        return {
            'n': summary['n'],
            f'median_{unit_suffix}': summary['median'],
            f'p16_{unit_suffix}': summary['p16'],
            f'p84_{unit_suffix}': summary['p84'],
            f'mode_{unit_suffix}': summary['mode'],
            'mode_bin': summary['mode_bin'],
            'counts': summary['counts'],
            'bin_edges': summary['bin_edges']
        }

    #Use of the above results
    events = _ensure_utc_index(events.copy(), 'events')
    feat = _ensure_utc_index(feat.copy(), 'feat')

    for col, msg in [
        (sun_col, "Sun altitude"),
        (wind_speed_col, "wind speed"),
        (wind_dir_col, "wind direction"),
        (dome_az_col, "dome azimuth"),
    ]:
        if col not in feat.columns:
            raise KeyError(f"feat must contain column '{col}' ({msg}).")

    sun = feat[sun_col].sort_index()

    # delta_min_start if missing
    if delta_min_start_col not in events.columns:
        if 't_iq_start' not in events.columns:
            raise KeyError(f"events must have '{delta_min_start_col}' or 't_iq_start'.")
        events[delta_min_start_col] = (events['t_iq_start'] - events.index).total_seconds() / 60.0

    # Compute/normalize t_iq_start series (for wind at IQ start)
    if 't_iq_start' in events.columns:
        t_iq_start_series = pd.to_datetime(events['t_iq_start'])
        if t_iq_start_series.dt.tz is None:
            t_iq_start_series = t_iq_start_series.dt.tz_localize('UTC')
        else:
            t_iq_start_series = t_iq_start_series.dt.tz_convert('UTC')
    else:
        t_iq_start_series = events.index + pd.to_timedelta(pd.to_numeric(events[delta_min_start_col], errors='coerce'), unit='m')

    #Sunset definitions
    sunset_times = _crossings_down(sun, 0.0)
    minus12_times = _crossings_down(sun, -12.0)

    t_open = events.index
    open_minus_sunset = []
    open_minus_minus12 = []

    for t in t_open:
        ss = _last_on_or_before(sunset_times, t)
        if pd.isna(ss):
            open_minus_sunset.append(np.nan)
            open_minus_minus12.append(np.nan)
            continue
        # sunset diff
        dt_ss_min = (t - ss).total_seconds() / 60.0
        open_minus_sunset.append(dt_ss_min)
        # -12° after that sunset
        m12_after_ss = _first_on_or_after(minus12_times, ss)
        if pd.isna(m12_after_ss):
            # fallback: nearest -12° to t
            i = minus12_times.searchsorted(t, side='left')
            cand = []
            if i > 0:
                cand.append(minus12_times[i-1])
            if i < len(minus12_times):
                cand.append(minus12_times[i])
            m12_after_ss = min(cand, key=lambda x: abs((t - x).total_seconds())) if cand else pd.NaT
        dt_m12_min = (t - m12_after_ss).total_seconds() / 60.0 if pd.notna(m12_after_ss) else np.nan
        open_minus_minus12.append(dt_m12_min)

    events = events.assign(
        open_minus_sunset_min=np.array(open_minus_sunset, dtype=float),
        open_minus_minus12_min=np.array(open_minus_minus12, dtype=float),
    )


    #Sun deg at open
    open_sun_alt = sun.reindex(t_open, method='nearest',
                               tolerance=pd.Timedelta(sun_match_tolerance))

    # ΔT0 and Δh0 at open
    dT_col = next((c for c in dT_col_candidates if c in feat.columns), None)
    dH_col = next((c for c in dH_col_candidates if c in feat.columns), None)
    dT0 = feat[dT_col].reindex(t_open, method='nearest',
                               tolerance=pd.Timedelta(env_match_tolerance)) if dT_col else pd.Series(index=t_open, dtype=float)
    dH0 = feat[dH_col].reindex(t_open, method='nearest',
                               tolerance=pd.Timedelta(env_match_tolerance)) if dH_col else pd.Series(index=t_open, dtype=float)

    # Wind speed/direction at open and at IQ start 
    t_iq_valid = pd.to_datetime(t_iq_start_series, errors='coerce')

    wind_speed_open = feat[wind_speed_col].reindex(t_open, method='nearest',
                                                   tolerance=pd.Timedelta(wind_match_tolerance))
    wind_speed_iq = feat[wind_speed_col].reindex(t_iq_valid, method='nearest',
                                                 tolerance=pd.Timedelta(wind_match_tolerance))
    wind_speed_iq.index = events.index

    wind_dir_open = feat[wind_dir_col].reindex(t_open, method='nearest',
                                               tolerance=pd.Timedelta(wind_match_tolerance))
    wind_dir_iq = feat[wind_dir_col].reindex(t_iq_valid, method='nearest',
                                             tolerance=pd.Timedelta(wind_match_tolerance))
    wind_dir_iq.index = events.index

    # Dome azimuth at open
    dome_az_open = feat[dome_az_col].reindex(t_open, method='nearest',
                                             tolerance=pd.Timedelta(dome_az_match_tolerance))

    # Summary
    h_delta = _rename_with_unit(_hist_summary(events[delta_min_start_col], bin_size_min), 'min')
    h_sunset = _rename_with_unit(_hist_summary(events['open_minus_sunset_min'], bin_size_min), 'min')
    h_m12 = _rename_with_unit(_hist_summary(events['open_minus_minus12_min'], bin_size_min), 'min')

    h_open_alt = _rename_with_unit(_hist_summary(open_sun_alt, bin_size_deg), 'deg')
    h_dT = _rename_with_unit(_hist_summary(dT0, bin_size_dT), 'C') if dT_col else None
    h_dH = _rename_with_unit(_hist_summary(dH0, bin_size_dH), 'kJkg') if dH_col else None

    h_alt_extras = {}
    for a, vals in open_minus_alt.items():
        h_alt_extras[a] = _rename_with_unit(_hist_summary(pd.Series(vals, index=t_open), bin_size_min), 'min')

    h_wind_speed_open = _rename_with_unit(_hist_summary(wind_speed_open, bin_size_wind), 'mps')
    h_wind_speed_iq = _rename_with_unit(_hist_summary(wind_speed_iq, bin_size_wind), 'mps')

    h_wind_dir_open = _hist_summary_direction(wind_dir_open, bin_size_dir_deg)
    h_wind_dir_iq = _hist_summary_direction(wind_dir_iq, bin_size_dir_deg)

    h_dome_az_open = _hist_summary_direction(dome_az_open, bin_size_dir_deg)

    summary = {
        'delta_min_start': h_delta,
        'open_minus_sunset_min': h_sunset,
        'open_minus_minus12_min': h_m12,
        'open_minus_altitude_min': h_alt_extras,  # dict keyed by altitude (deg)
        'open_sun_altitude_deg': h_open_alt,
        'dT0_C': h_dT,
        'dH0_kJkg_moist': h_dH,
        'wind_speed_open_mps': h_wind_speed_open,
        'wind_speed_iq_start_mps': h_wind_speed_iq,
        'wind_dir_open_deg': h_wind_dir_open,
        'wind_dir_iq_start_deg': h_wind_dir_iq,
        'dome_az_open_deg': h_dome_az_open,
    }

    figs = []
    if show or return_figs:
        # Main figure: 2x3 grid
        fig, axs = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

        def _plot_hist(ax, entry, title, xlabel, zero_line=False):
            if entry is None or entry.get('n', 0) == 0:
                ax.set_title(title + ' (no data)')
                ax.set_xlabel(xlabel)
                ax.set_ylabel('Count')
                ax.grid(True, alpha=0.3)
                return
            counts, edges = entry['counts'], entry['bin_edges']
            centers = (edges[:-1] + edges[1:]) / 2.0
            ax.hist(centers, bins=edges, weights=counts, alpha=0.85, color='tab:blue', edgecolor='white')
            if zero_line:
                ax.axvline(0, color='k', lw=0.8, alpha=0.8)
            # Mode line (linear or circular)
            mode_key = next((k for k in entry.keys() if k.startswith('mode_') or k == 'mode'), None)
            if mode_key and np.isfinite(entry.get(mode_key, np.nan)):
                ax.axvline(entry[mode_key], color='tab:red', ls='--', lw=1.2, label=f"mode ≈ {entry[mode_key]:.2f}")
            # Circular mean line if provided
            #if 'circ_mean_deg' in entry and np.isfinite(entry.get('circ_mean_deg', np.nan)):
            #    ax.axvline(entry['circ_mean_deg'], color='tab:orange', ls=':', lw=1.2, label=f"circ mean ≈ {entry['circ_mean_deg']:.1f}°")
            #if 'circ_std_deg' in entry and np.isfinite(entry.get('circ_std_deg', np.nan)):
                #ax.set_title(f"{title}\n(circ std ≈ {entry['circ_std_deg']:.1f}°)")
            ax.set_title(f"{title}")
            ax.legend(loc='best')
            ax.set_xlabel(xlabel)
            ax.set_ylabel('Count')
            ax.grid(True, alpha=0.3)

        _plot_hist(axs[0, 0], h_delta, 'IQ(t0) − Dome open(to)', 'Minutes', zero_line=True)
        _plot_hist(axs[0, 1], h_sunset, 't_open − sunset (0°)', 'Minutes', zero_line=True)
        _plot_hist(axs[0, 2], h_m12, 't_open − (−12°)', 'Minutes', zero_line=True)

        _plot_hist(axs[1, 0], h_open_alt, 'Sun altitude at open', 'Degrees')
        _plot_hist(axs[1, 1], h_dT, 'ΔT at open', '°C', zero_line=True)
        _plot_hist(axs[1, 2], h_dH, 'Δh at open', 'kJ/kg moist', zero_line=True)

        if show:
            plt.show()
        figs.append(fig)

        # Extra altitude panels (if any)
        if len(h_alt_extras) > 0:
            n = len(h_alt_extras)
            ncols = min(3, n)
            nrows = int(np.ceil(n / ncols))
            fig2, axs2 = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows), constrained_layout=True)
            axs2 = np.atleast_1d(axs2).ravel()
            for ax, (a, entry) in zip(axs2, sorted(h_alt_extras.items(), key=lambda kv: kv[0])):
                _plot_hist(ax, entry, f't_open − ({a:.0f}°)', 'Minutes', zero_line=True)
            for ax in axs2[n:]:
                ax.set_visible(False)
            if show:
                plt.show()
            figs.append(fig2)

        # Wind speed figure (1x2)
        fig3, axs3 = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
        _plot_hist(axs3[0], h_wind_speed_open, 'Wind speed at open', 'm/s')
        _plot_hist(axs3[1], h_wind_speed_iq, 'Wind speed at IQ start', 'm/s')
        if show:
            plt.show()
        figs.append(fig3)

        # Direction figure (1x3): wind @ open, wind @ IQ start, dome az @ open
        fig4, axs4 = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
        _plot_hist(axs4[0], h_wind_dir_open, 'Wind direction at open', 'Degress')
        _plot_hist(axs4[1], h_wind_dir_iq, 'Wind direction at IQ start', 'Degress')
        _plot_hist(axs4[2], h_dome_az_open, 'Dome azimuth at open', 'Degress')
        if show:
            plt.show()
        figs.append(fig4)

    return (summary, figs if return_figs else None)

For documentation a report is printed:

In [ ]:
# Run the report
summary, figs = report_open_timing_histograms(
    events=events_with_tno,
    feat=features,
    sun_col='SunAltitude',
    wind_speed_col='WindSpeed_301',
    wind_dir_col='WindDirection_301',
    dome_az_col='Dome_Azimuth_Actual',
    delta_min_start_col='delta_min_start',  
    altitudes_deg=(-6.0, -18.0),
    bin_size_min=5.0,
    bin_size_deg=1.0,
    bin_size_dT=0.2,
    bin_size_dH=1.0,
    bin_size_wind=0.5,
    bin_size_dir_deg=10.0,
    sun_match_tolerance='10min',
    env_match_tolerance='10min',
    wind_match_tolerance='10min',
    dome_az_match_tolerance='10min',
    show=True,
    return_figs=True
)

# Read a few results
print("Δstart median (min):", summary['delta_min_start']['median_min'])
print("Mode t_open − sunset (min):", summary['open_minus_sunset_min']['mode_min'])
print("Mode t_open − (−12°) (min):", summary['open_minus_minus12_min']['mode_min'])

# Wind speed
print("Wind speed @ open median (m/s):", summary['wind_speed_open_mps']['median_mps'])
print("Wind speed @ IQ-start median (m/s):", summary['wind_speed_iq_start_mps']['median_mps'])

# Wind direction (circular stats)
wd_open = summary['wind_dir_open_deg']
wd_iq   = summary['wind_dir_iq_start_deg']
print("Wind dir @ open: circ mean (deg) =", wd_open['circ_mean_deg'],
      ", circ std (deg) =", wd_open['circ_std_deg'],
      ", mode bin center (deg) =", wd_open['mode_deg'])
print("Wind dir @ IQ-start: circ mean (deg) =", wd_iq['circ_mean_deg'],
      ", circ std (deg) =", wd_iq['circ_std_deg'],
      ", mode bin center (deg) =", wd_iq['mode_deg'])

# Dome azimuth at open (circular stats)
daz = summary['dome_az_open_deg']
print("Dome az @ open: circ mean (deg) =", daz['circ_mean_deg'],
      ", circ std (deg) =", daz['circ_std_deg'],
      ", mode bin center (deg) =", daz['mode_deg'])

if figs:
    for i, fig in enumerate(figs, start=1):
        fig.savefig(f"open_timing_histograms_{i}.png", dpi=130)

In [ ]:
#Reproducibility
print_session_info()